<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 Viggle Remake: Recast or Edit Any Video</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition | Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Swap people or edit anything · Motion, camera and soundtrack kept | MiniMax H3 20B INT8 | Powered by Wan2GP</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

**[Viggle Remake](https://huggingface.co/Viggle/Viggle-Remake)** re-renders a video: repaint one to four of its frames, describe the finished clip, and it carries the change through the whole video while keeping the **original motion, camera and soundtrack**. Built on **MiniMax H3 20B**, with two modes:

- 🧑‍🎤 **Swap**: recast people and characters (face, hair, build, outfit)
- 🪄 **Edit**: change objects, setting, weather, light or style; the people stay

**Quick start:** run **Step 1** and **Step 2**, open the link Step 2 prints, click an example under **📚 Examples** at the bottom of the page and press **🎬 Remake Video**. The ⚡ quick test takes about 6 minutes.


## ⚙️ Choose the mode (first line of Step 1)

Set `REMAKE_MODE` at the top of **Step 1** to `"swap"` or `"edit"`, then run the cells. Step 1 downloads only that mode's 21 GB transformer (plus the shared ~18 GB text encoder and VAEs). To switch, change the line and re-run Step 1 and Step 2.

## 🖌️ How to make a good remake

1. **Pick the window**: the start time and length in the source. One render is up to 14.4 s; 480p and 4-6 s are the sweet spot on T4.
2. **Grab a frame** (🎞️ in the UI) where the subject is large, sharp and facing the camera. Download it.
3. **Paint it** in any image editor (ChatGPT / Gemini image editing, Qwen Image Edit, Photoshop...). Keep the **same size, framing and pose**; change only what should change. Example instruction: *"Replace the man with a woman with a long black braid in a mustard-yellow overall. Keep his pose, the background, the framing and the image size."*
4. **Drop it into Painted frame 1**. The frame grabber already wrote its frame number into the pin. Add a second painted frame only when a character turns to show another side, someone new enters, or a shot runs longer than ~8 s.
5. **Pin** the painted frames (on by default in Swap): the video passes through each one at its frame. In Swap, pin every frame, or the source person can come back. In Edit, pin when the clip has cuts or the exact composition matters.
6. **Prompt**: fill the ✍️ builder and press *Build*, or write the six sections yourself (`subject_definitions`, `summary`, `retention_analysis`, `detailed_description`, `overall_soundscape`, `non_diegetic_music`). Describe the **finished** video, not instructions. Never write "dim" (it darkens the whole render).

## ⏱️ How long it takes on 2 x T4

| Render | ⚡ Fast (default) | Exact |
|---|---|---|
| 480p · 3.8 s | ~5 min | ~8 min |
| 480p · 5.2 s | **~6 min** | ~12 min |
| 480p · 7.9 s | ~11 min | ~23 min |
| 480p · 9.4 s | ~14 min | ~32 min |
| 720p · 5.2 s | ~26 min | ~60 min |

**⚡ Attention** (next to the seed): *Fast* uses Comfy Kitchen's INT8 attention for the T4 and is about twice as fast; *Exact* is the full-precision reference.

> **Responsible use:** the identity comes from the frames you paint. Only change footage you have the rights to, never put a real person into a video without their agreement, and label results as AI-generated.


## 📦 Step 1: Environment, Wan2GP and model downloads

Installs Wan2GP and the packages it needs, then downloads ~39 GB of weights (about 5-8 minutes). Files you attach as a Kaggle Dataset are used instead of downloading.

In [ ]:
# Step 1: environment, pinned Wan2GP, dependencies and model downloads

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  👉 CHOOSE THE MODE: "swap" (default) or "edit"                                  ║
# ║     swap: recast people and characters (face, hair, outfit, whole body)          ║
# ║     edit: remove / add / replace objects, change setting, weather, light, style  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
REMAKE_MODE = "swap"

import os
import sys
import time
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~23 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_viggle.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# ---- Settings (every source is locked to one version) ----
WAN2GP_COMMIT = "38758d3c73366422efcf609abb63cab8385f3552"   # deepbeepmeep/Wan2GP, 12 Nov 2026 (the version this notebook is built on)
VIGGLE_REVISION = "7e5be143702dd53b9154482de673b8519ed2d4d6"  # Viggle/Viggle-Remake, 7 Nov 2026
H3_REVISION = "fe905e15831c620e804ba9e9382e4d9b4f60b21e"      # DeepBeepMeep/MiniMax-H3 (text encoder, VAEs)
MODELS_DIR = "/kaggle/tmp/viggle_models" if os.path.isdir("/kaggle") else os.path.abspath("viggle_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of 40 GB in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODE_MENU = {"swap": "recast people and characters", "edit": "change objects, setting, weather or light"}
if REMAKE_MODE not in MODE_MENU:
    raise SystemExit(f'❌ Unknown REMAKE_MODE "{REMAKE_MODE}". Set it to "swap" or "edit".')
try:
    gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
except Exception:
    gpus = []
ram_gb = psutil.virtual_memory().total / 1024**3
if len(gpus) < 2:
    print("❌ This notebook needs 2 GPUs: Settings → Accelerator → GPU T4 x2")
print(f"✅ Mode: {REMAKE_MODE} ({MODE_MENU[REMAKE_MODE]}) | {len(gpus)} x {gpus[0] if gpus else 'no GPU'} | RAM {ram_gb:.0f} GB")

# ---- 1. Wan2GP, fetched by commit (never a branch) ----
t0 = time.time()
if not os.path.isdir("Wan2GP/.git"):
    subprocess.run(["git", "init", "-q", "Wan2GP"], check=True)
    subprocess.run(["git", "-C", "Wan2GP", "remote", "add", "origin", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
subprocess.run(["git", "-C", "Wan2GP", "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT], check=True)
subprocess.run(["git", "-C", "Wan2GP", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits on re-runs
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]} ({time.time() - t0:.0f}s)")

# ---- 2. Dependencies: only what the H3 / Viggle Remake path imports ----
# (Wan2GP's full requirements.txt pins an onnxruntime nightly that is not on PyPI.)
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
                "comfy-kitchen==0.2.35", "transformers==4.54.0", "diffusers==0.36.0", "gguf==0.17.1", "optimum-quanto==0.2.7",
                "accelerate>=1.1.1", "gradio==5.29.0", "huggingface_hub[hf_xet]", "hf_xet>=1.5.2", "sentencepiece", "soundfile",
                "ffmpeg-python", "imageio", "imageio-ffmpeg"], check=True)
print(f"✅ Dependencies installed ({time.time() - t0:.0f}s)")

# ---- 3. Model weights (parallel; files from an attached Kaggle Dataset are reused) ----
H3, VIGGLE, TE = "DeepBeepMeep/MiniMax-H3", "Viggle/Viggle-Remake", "Qwen3-VL-32B-Instruct"
DIT = (VIGGLE, f"int8/viggle_remake_{REMAKE_MODE}_pruned_int8_convrot.safetensors", 20.97, VIGGLE_REVISION)
OTHER_DIT = os.path.join(MODELS_DIR, f"int8/viggle_remake_{'edit' if REMAKE_MODE == 'swap' else 'swap'}_pruned_int8_convrot.safetensors")
BIG = [DIT,
       (H3, f"{TE}/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf", 14.58, H3_REVISION),
       (H3, "minimax_h3/minimax_h3_video_vae_int8_convrot.safetensors", 2.81, H3_REVISION),
       (H3, "MiniMax-H3-audio_vae_fp32.safetensors", 0.61, H3_REVISION)]
SMALL = [(H3, f"{TE}/{name}", 0.0, H3_REVISION) for name in
         ("config.json", "tokenizer.json", "tokenizer_config.json", "vocab.json", "preprocessor_config.json", "chat_template.json")]
EXAMPLE = {"swap": "swap-pineapple-mascot", "edit": "edit-remove-turbines"}[REMAKE_MODE]  # Viggle's official example
SMALL += [(VIGGLE, f"examples/inputs/{EXAMPLE}/{name}", 0.0, VIGGLE_REVISION) for name in ("source.mp4", "ref1.png", "ref2.png")]
SMALL += [(VIGGLE, f"examples/prompts/{EXAMPLE}.txt", 0.0, VIGGLE_REVISION)]
os.makedirs(MODELS_DIR, exist_ok=True)

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            attached.setdefault(f, os.path.join(root, f))


def fetch(item):
    from huggingface_hub import hf_hub_download
    repo, filename, size_gb, revision = item
    dest = os.path.join(MODELS_DIR, filename)
    if os.path.exists(dest):
        return f"✓ {filename}"
    source = attached.get(os.path.basename(filename)) if size_gb else None
    if source and os.path.getsize(source) > 0:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir=MODELS_DIR)
            return f"✅ {filename}" + (f" ({size_gb:.1f} GB)" if size_gb else "")
        except Exception as e:
            if attempt == 3:
                raise
            say(f"⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


todo = [item for item in BIG if not os.path.exists(os.path.join(MODELS_DIR, item[1])) and os.path.basename(item[1]) not in attached]
need_gb = sum(item[2] for item in todo)
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
if need_gb > free_gb - 2 and os.path.exists(OTHER_DIT):
    os.remove(OTHER_DIT)  # the other mode's 21 GB transformer: room for this one (Step 1 downloads it again when you switch back)
    print(f"🧹 Removed the other mode's transformer ({os.path.basename(OTHER_DIT)}) to make room")
    free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
if need_gb > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need_gb:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA (bytes on disk, which hf_xet fills as it goes)
    base = folder_bytes(MODELS_DIR)
    total = max(need_gb * 1e9, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=6) as pool:
        for job in [pool.submit(fetch, item) for item in BIG + SMALL]:
            job.result()
finally:
    done.set()
    time.sleep(1.1)
if need_gb:
    say(f"✅ {need_gb:.1f} GB downloaded in {fmt_time(time.time() - t0)} (avg {need_gb * 1e3 / max(time.time() - t0, 1):.0f} MB/s)")
shutil.rmtree(os.path.join(MODELS_DIR, ".cache"), ignore_errors=True)

# Settings for Step 2's engine (also saved to a file, so Step 2 can be re-run alone after a kernel restart)
import json
VR_SETTINGS = {"VR_MODE": REMAKE_MODE, "VR_MODELS_DIR": MODELS_DIR, "VR_DIT_FILE": DIT[1],
               "VR_EXAMPLES_DIR": os.path.join(MODELS_DIR, "examples")}
os.environ.update(VR_SETTINGS)
with open("viggle_settings.json", "w") as f:
    json.dump(VR_SETTINGS, f)
print(f"✅ All weights ready. Run Step 2 (disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB)")


## 🚀 Step 2: Engine + Gradio UI

Starts the engine: model load (~3 min), then the **Cloudflare** and **Gradio** public links. Open either one. If the Cloudflare link is still warming up, it appears in the cell a minute later.

**In the app:** upload a source video → 🎞️ grab a frame → paint it elsewhere → drop it into *Painted frame 1* → ✍️ build the prompt → 🎬 **Remake Video**. The progress bar at the top of the page keeps working if the connection drops, and *📂 Load latest video* fetches the result at any time.

In [ ]:
# Step 2: Engine + branded Gradio UI
# Writes run_viggle.py, then runs it: model load (~3 min) → public links.
import codecs
import os
import re
import sys
import subprocess

RUN_SCRIPT_CODE = r'''"""Viggle Remake (Swap / Edit) video-to-video engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_viggle.py and runs it. It drives Wan2GP's MiniMax H3 / Viggle Remake code:

* DiT: Viggle's merged checkpoint for the chosen mode (MiniMax H3 FL2VA pruned 20B with the Swap or Edit LoRA merged
  in FP32 and then quantized once to INT8 ConvRot). No LoRA runs at generation time.
* Every INT8 linear runs on Comfy Kitchen's INT8 tensor-core GEMM (W8A8 with the ConvRot Hadamard rotation).
* Tensor-parallel DiT: every block is split in half across the two T4s (attention heads and FFN columns), so both GPUs
  compute every block at the same time and only exchange the block's partial outputs. Nothing is streamed per step.
  Each GPU's half is queued from a CPU thread of its own, so neither GPU waits for the other's kernel launches.
* Attention: Comfy Kitchen's INT8 attention for the T4 by default (about 2x faster steps, ~1 % from exact, checked
  against exact attention on the user's video in step 1 of every run), or exact PyTorch SDPA (FP16) in the UI.
* Qwen3-VL-32B (GGUF Q4_K_M) and the VAEs are pinned in RAM and run by MMGP on GPU 1, unloaded before denoising.
  The VAE decoder runs on both GPUs (alternate clips).
"""
import copy
import faulthandler
import functools
import gc
import glob
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
import warnings

warnings.filterwarnings("ignore")
faulthandler.enable(file=sys.__stderr__, all_threads=True)  # a hard crash (segfault, abort) prints where it happened
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

WAN2GP_DIR = os.path.abspath("Wan2GP")
WORK_DIR = os.path.dirname(WAN2GP_DIR)
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
MODELS_DIR = os.environ.get("VR_MODELS_DIR", "/kaggle/tmp/viggle_models")
EXAMPLES_DIR = os.environ.get("VR_EXAMPLES_DIR", os.path.join(MODELS_DIR, "examples"))
MODE = os.environ.get("VR_MODE", "swap")
DIT_FILE = os.environ.get("VR_DIT_FILE", f"viggle_remake_{MODE}_pruned_int8_convrot.safetensors")
TE_FILE = os.environ.get("VR_TE_FILE", "qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf")
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

import numpy as np
import psutil
import torch
from PIL import Image

import gradio as gr


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _ram_in_use():
    """total - available: unlike psutil's 'used', this counts pinned (page-locked) memory, which Linux files as shared."""
    vm = psutil.virtual_memory()
    return f"{_gb(vm.total - vm.available)} / {_gb(vm.total)}"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites it in place).
    A finished phase just clears its line (minimal logs: callers print one summary line where it matters). Step-based phases (denoising) interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between callbacks."""

    WIDTH, LINE = 24, 104
    WATCHED = {"decode": 1800, "stitch": 600, "audio": 600, "save": 600}  # phase -> seconds before a stack dump

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.drawn = 0  # characters of the live line on screen
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            if self.phase is not None and self.phase["key"] == key:
                return
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            if key in self.WATCHED:  # a phase that should take seconds: dump every thread's stack if it hangs
                faulthandler.dump_traceback_later(self.WATCHED[key], file=sys.__stderr__)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.flush()

    def abort(self):
        """End the live line of a phase that failed (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            faulthandler.cancel_dump_traceback_later()
            self._clear()
            self.out.write(f"  ❌ {p['label']} failed after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<16} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.1f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        text = self._render()
        self.drawn = max(self.drawn, len(text), self.LINE)
        self.out.write("\r" + text.ljust(self.drawn))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.drawn + "\r")
            self.visible = False
            self.drawn = 0

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)



if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise SystemExit("❌ This notebook needs Kaggle's 'GPU T4 x2' accelerator (Settings → Accelerator → GPU T4 x2).")

DIT_DEV = torch.device("cuda", 0)   # one half of every DiT block
MAIN_DEV = torch.device("cuda", 1)  # the other half + text encoder + VAEs + sampler state
torch.cuda.set_device(MAIN_DEV)     # Wan2GP / MMGP use the current device for "cuda"
DEVICES = (DIT_DEV, MAIN_DEV)

torch.backends.cuda.enable_flash_sdp(False)       # Turing (SM 7.5) has no flash kernel
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = False

_props = torch.cuda.get_device_properties(0)
MODE_LABEL = {"swap": "🧑‍🎤 Swap: recast people and characters", "edit": "🪄 Edit: objects, setting, weather, light"}[MODE]


def _on(dev):
    """torch.cuda.device(dev) for a CUDA device; nothing for the CPU."""
    import contextlib
    return torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()


# Turing has no BF16 convolution kernels. Convolutions that reach BF16 run in FP32.
if _props.major < 8:
    def _conv_fp32_on_turing(original):
        def forward(self, input, weight, bias):
            if input.is_cuda and input.dtype == torch.bfloat16:
                bias32 = None if bias is None else bias.float()
                return original(self, input.float(), weight.float(), bias32).to(torch.bfloat16)
            return original(self, input, weight, bias)
        return forward
    for _conv in (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Conv3d):
        _conv._conv_forward = _conv_fp32_on_turing(_conv._conv_forward)

# ==== Wan2GP runtime: quantization handlers + INT8 kernels (what wgp.py does at import) ====
# Wan2GP's shared utilities import rembg (background removal, which pulls in onnxruntime) at module level. Viggle
# Remake never removes backgrounds, so a stand-in module is registered when rembg is not installed.
import types as _types

try:
    import rembg  # noqa: F401
except Exception:
    _rembg = _types.ModuleType("rembg")

    def _no_rembg(*_args, **_kwargs):
        raise RuntimeError("Background removal (rembg) is not installed in this notebook")

    _rembg.remove = _rembg.new_session = _no_rembg
    sys.modules["rembg"] = _rembg

from mmgp import offload, quant_router

quant_router.unregister_handler(".fp8_quanto_bridge")
for _handler in ("shared.qtypes.int8_convrot", "shared.qtypes.gguf"):  # DiT + VAE decoder, Qwen3-VL GGUF
    quant_router.register_handler(_handler)
from shared.qtypes import gguf as _gguf_handler

quant_router.register_file_extension("gguf", _gguf_handler)

from shared.kernels import int8_backend

try:
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using the PyTorch path: {e}")
# Kitchen splits each INT8 GEMM into row tiles that fit a 16 MB scratch buffer, which on a T4 means 64-row GEMMs
# for the 28672-wide FFN. 128 MB gives ~700-row tiles: far better tensor-core use for ~100 MB of extra scratch.
int8_backend._SCRATCH_BYTES = 128 * 1024 * 1024
INT8_LABEL = {"kitchen": "Comfy Kitchen INT8 tensor cores", "wgpkernels": "WGPKernels INT8", "triton": "Triton INT8 kernels"}.get(
    int8_backend._backend, "PyTorch INT8 fallback (slow)")
offload.shared_state["_attention"] = "sdpa"

from shared.utils import files_locator as fl

fl.set_checkpoints_paths([MODELS_DIR, "ckpts", "."])

import models.minimax_h3.minimax_h3_main as h3_main
from models.minimax_h3 import pipeline as h3_pipeline
from models.minimax_h3.minimax_h3_handler import family_handler
from models.minimax_h3.minimax_h3_main import VIDEO_VAE_INT8_FILE
from models.minimax_h3.viggle import VIGGLE_REMAKE_MAX_FRAMES, VIGGLE_REMAKE_MIN_FRAMES, VIGGLE_REMAKE_SHIFT, VIGGLE_REMAKE_STEPS

AUDIO_RATE = h3_pipeline.AUDIO_SAMPLE_RATE

# ComfyUI-layout pruned checkpoints may also carry the AdaLN basis / mean they were compressed with. Wan2GP's model has
# no such tensors, so drop them before loading.
_orig_strip_wrappers = h3_main._strip_wrappers


def _strip_wrappers(state_dict, *args, **kwargs):
    for key in [k for k in state_dict if k.endswith(("adaln_basis", "adaln_mean"))]:
        state_dict.pop(key)
    return _orig_strip_wrappers(state_dict, *args, **kwargs)


h3_main._strip_wrappers = _strip_wrappers


# ==== Text encoder compute dtype ====
# Without llama.cpp kernels (no wheel for Kaggle's Python), each GGUF linear is dequantized to a dense weight and run
# as a normal matmul. Turing has no BF16 tensor cores, so a BF16 GEMM can be slower than plain FP32. Measure both on
# the real layer shape and keep the faster one (FP32 is also the more precise of the two).
def _gemm_seconds(dtype, m=512, k=5120, n=25600, reps=3):
    a = torch.randn(m, k, device=MAIN_DEV, dtype=dtype)
    b = torch.randn(n, k, device=MAIN_DEV, dtype=dtype)
    torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    t = time.time()
    for _ in range(reps):
        torch.nn.functional.linear(a, b)
    torch.cuda.synchronize(MAIN_DEV)
    return (time.time() - t) / reps


try:
    _bf16_s, _fp32_s = _gemm_seconds(torch.bfloat16), _gemm_seconds(torch.float32)
    TE_DTYPE = torch.float32 if _fp32_s < _bf16_s else torch.bfloat16
except Exception:
    TE_DTYPE = torch.bfloat16
torch.cuda.empty_cache()
_orig_load_text_encoder = h3_main._load_text_encoder
h3_main._load_text_encoder = lambda filename, dtype, distilled=False: _orig_load_text_encoder(filename, TE_DTYPE, distilled)

# ==== Load the model ====
MODEL_TYPE = "viggle_remake"
_t_load = time.time()
BAR.start("load", "Loading models (~3 min)")
model_def = {"architecture": MODEL_TYPE, "viggle_remake_mode": MODE}
model_def.update(family_handler.query_model_def(MODEL_TYPE, model_def))
model_def["viggle_remake_mode"] = MODE
model_def["qkv_splitting"] = False             # keep the fused QKV GEMM: bigger tiles, and the tensor-parallel split below
model_def["video_vae_file"] = VIDEO_VAE_INT8_FILE  # INT8 ConvRot VAE decoder: half the VRAM, Kitchen GEMMs
pipeline, _ = family_handler.load_model(
    model_filename=[os.path.join(MODELS_DIR, DIT_FILE)],
    model_type=MODEL_TYPE,
    base_model_type=MODEL_TYPE,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=os.path.join(MODELS_DIR, "Qwen3-VL-32B-Instruct", TE_FILE),
)
if pipeline.remake_mode != MODE:
    raise SystemExit(f"❌ The pipeline was built for '{pipeline.remake_mode}', expected '{MODE}'")
transformer = pipeline.transformer
transformer.cache = None

# Modules H3 pins to FP32 (patch projections, output heads, AdaLN curve projections) are converted here,
# since this transformer is not handed to MMGP's profile (which would otherwise apply the locks).
for _module in transformer.modules():
    _lock = getattr(_module, "_lock_dtype", None)
    if _lock is not None and isinstance(_module, torch.nn.Linear) and _module.weight.dtype.is_floating_point:
        _module.to(_lock)

# ==== Tensor-parallel DiT: each block is split in half across the two T4s ====
# Within a block, GPU 0 computes attention heads 0-27 and FFN columns 0-7167, GPU 1 the other half. Both start from
# the same residual stream (kept as a replica on each GPU), and after attention and after the FFN each GPU sends its
# partial output projection to the other, so both GPUs hold the full sum. The weight slices follow the ConvRot
# layout exactly: QKV / FFN-up are split by output rows (Q, K, V are grouped), output / FFN-down by input columns on
# 256-wide rotation-group boundaries, so every INT8 GEMM computes precisely its share of the full product.
import models.minimax_h3.transformer as _h3_transformer
from models.minimax_h3.transformer import MLP as H3MLP, Attention as H3Attention, _gated_residual, _norm_modulate, _take
from shared.qtypes.int8_convrot import _rotate_activation

_tp_cache = {"rope": None, "rope_copy": None, "temb": None, "temb_copy": None, "src": None, "replica": None}


# >>> section:tp
class Int8Shard(torch.nn.Module):
    """One GPU's slice of an INT8 ConvRot linear (a set of output rows, or a range of input columns)."""

    def __init__(self, linear, device, rows=None, cols=None):
        super().__init__()
        data, scale = linear.weight._data, linear.weight._scale.reshape(-1)
        if rows is not None:
            data, scale = data.index_select(0, rows), scale.index_select(0, rows)
        if cols is not None:
            data = data[:, cols]
        # copy=True: own storage even on the same device (never a view into the memory-mapped checkpoint)
        self.register_buffer("qdata", data.contiguous().to(device, copy=True), persistent=False)
        self.register_buffer("qscale", scale.float().contiguous().to(device, copy=True), persistent=False)
        self.group = int(getattr(linear, "_convrot_group_size", 0) or 0)
        self.out_features, self.in_features = self.qdata.shape

    def forward(self, x):
        x2 = x.reshape(-1, self.in_features)
        if int8_backend.kitchen_enabled() and self.group == 256 and x2.is_cuda and self.out_features % 4 == 0:
            out = int8_backend._linear_impl(x2, self.qdata, self.qscale, None, True)
        else:  # reference path (no Kitchen): rotate, dequantize, matmul
            weight = self.qdata.to(x2.dtype) * self.qscale.to(x2.dtype)[:, None]
            out = torch.nn.functional.linear(_rotate_activation(x2, self.group), weight)
        return out.reshape(*x.shape[:-1], self.out_features)


def _copy_to(value, dev, slot):
    """Per-forward copies of temb / rope for GPU 0 (they are the same tensor object for all 50 blocks)."""
    if value is None:
        return None
    first = value[0] if isinstance(value, tuple) else value
    if first.device == dev:
        return value
    if _tp_cache[slot] is not first:
        _tp_cache[slot] = first
        _tp_cache[slot + "_copy"] = tuple(v.to(dev) for v in value) if isinstance(value, tuple) else value.to(dev)
    return _tp_cache[slot + "_copy"]


def _all_reduce_residual(hiddens, partials, gates, segments):
    """hidden += gate * (partial_0 + partial_1) on both GPUs."""
    incoming = [partials[1].to(DEVICES[0], non_blocking=True, copy=True),
                partials[0].to(DEVICES[1], non_blocking=True, copy=True)]
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            branch = partials[s].add_(incoming[s])
            out.append(_gated_residual([hiddens[s]], [gates[s]], [branch], segments))
    return out


# Long-video mode: the same block math on row slices. A normal block builds several full-length intermediates per GPU
# (modulated input, fused QKV, BF16 and FP16 copies of Q/K/V, the output-projection input, the exchanged partial sums):
# roughly 75-110 KB per token. For long or 720p clips that does not fit next to half of the model, so in this mode only
# what attention needs at full length is kept (the residual stream and FP16 Q/K/V, ~40 KB per token). Norms,
# projections, RoPE, the FFN and the GPU exchange run on 8192-row slices. Same arithmetic, a little more Python per block.
LONG_VIDEO = {"on": False, "rows": 8192}


def _row_chunks(n):
    step = LONG_VIDEO["rows"]
    return [(a, min(n, a + step)) for a in range(0, n, step)]


def _slice_segments(segments, a, b):
    """(start, stop, modality row) segments of the packed sequence, clipped to rows [a, b) and shifted to 0."""
    return [(max(s0, a) - a, min(s1, b) - a, row) for s0, s1, row in segments if min(s1, b) > max(s0, a)]


def _apply_rope(query, key, rope):
    """H3's split-half rotary embedding on [1, n, heads, dim] tensors, in place."""
    pairs = rope.shape[-2]
    cosine, sine = rope[..., 0], rope[..., 1]
    scratch = torch.empty_like(query[..., :pairs])
    for tensor in (query, key):
        first, second = tensor[..., :pairs], tensor[..., pairs:2 * pairs]
        scratch.copy_(first)
        first.mul_(cosine).addcmul_(second, sine, value=-1)
        second.mul_(cosine).addcmul_(scratch, sine)


def _sliced_attention(shard, hidden, mod, segments, rope):
    attn = shard.attn
    rows, heads, dim = hidden.shape[0], attn.heads, attn.head_dim
    inner = heads * dim
    if isinstance(rope, tuple):
        rope = rope[0]
    q, k, v = (torch.empty(1, rows, heads, dim, device=hidden.device, dtype=torch.float16) for _ in range(3))
    for a, b in _row_chunks(rows):
        h = _norm_modulate(shard.norm1, hidden[a:b], [mod[0], mod[1]], _slice_segments(segments, a, b))
        qkv = attn.qkv_proj(h)
        del h
        query, key, value = (t.reshape(1, b - a, heads, dim) for t in qkv.split(inner, dim=-1))
        query, key = attn.q_norm(query), attn.k_norm(key)
        if rope is not None:
            _apply_rope(query, key, rope[:, a:b])
        q[:, a:b].copy_(query)
        k[:, a:b].copy_(key)
        v[:, a:b].copy_(value)
        del qkv, query, key, value
    qkv_list = [q, k, v]
    del q, k, v
    out = _h3_transformer.pay_attention(qkv_list, recycle_q=True)[0]  # [rows, heads, dim] FP16
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(rows):
        partial[a:b] = attn.out_proj(out[a:b].reshape(b - a, inner).to(hidden.dtype))
    return partial


def _sliced_mlp(shard, hidden, mod, segments):
    partial = torch.empty_like(hidden)
    for a, b in _row_chunks(hidden.shape[0]):
        h = _norm_modulate(shard.norm2, hidden[a:b], [mod[3], mod[4]], _slice_segments(segments, a, b))
        partial[a:b] = shard.mlp([h])
    return partial


def _all_reduce_sliced(hiddens, partials, gates, segments):
    """_all_reduce_residual without full-length copies: the partial sums cross between GPUs slice by slice."""
    for a, b in _row_chunks(hiddens[0].shape[0]):
        incoming = [partials[1][a:b].to(DEVICES[0], non_blocking=True, copy=True),
                    partials[0][a:b].to(DEVICES[1], non_blocking=True, copy=True)]
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials[s][a:b].add_(incoming[s])
        del incoming
    out = []
    for s, dev in enumerate(DEVICES):
        with _on(dev):
            out.append(_gated_residual([hiddens[s]], [gates[s]], [partials[s]], segments))
    return out


# One CPU thread per GPU. With a single Python thread queuing both GPUs' kernels one after the other, anything that holds
# that thread up on one GPU (a full launch queue, a copy between the GPUs staged through host memory, an allocator flush)
# also holds back the other GPU's next kernels, and the other GPU sits idle. Each GPU has a thread of its own that queues
# its half of every block, and the two threads meet only where the math needs both halves: the two partial-sum exchanges
# per block. Same kernels in the same order on each GPU, so the result is
# bit-identical. The first transformer pass of the session runs on one thread (it compiles every kernel on both GPUs);
# the per-GPU threads take over from the second pass.
TP_THREADS = {"on": True, "warm": False}
TP_STATE = {"block": -1}  # index of the block being queued (read by the INT8 attention check)
_TP_BARRIER = threading.Barrier(len(DEVICES), timeout=1800)
_TP_POOLS = []


def _tp_pools():
    if not _TP_POOLS:
        from concurrent.futures import ThreadPoolExecutor
        for dev in DEVICES:
            init = (lambda d=dev: torch.cuda.set_device(d)) if dev.type == "cuda" else None
            _TP_POOLS.append(ThreadPoolExecutor(1, thread_name_prefix=f"tp{len(_TP_POOLS)}", initializer=init))
    return _TP_POOLS


def _tp_job(fn, s, grad, inference, args):
    try:
        with torch.inference_mode(inference), torch.set_grad_enabled(grad), _on(DEVICES[s]):
            return fn(s, *args)
    except BaseException:
        _TP_BARRIER.abort()  # the other GPU's thread stops waiting at its next exchange
        raise


def _run_on_both(fn, args_per_gpu):
    """fn(s, *args) on every GPU's own thread at once. Returns when both threads have queued their kernels, not when the
    GPUs are done with them."""
    if _TP_BARRIER.broken:
        _TP_BARRIER.reset()
    grad, inference = torch.is_grad_enabled(), torch.is_inference_mode_enabled()
    futures = [pool.submit(_tp_job, fn, s, grad, inference, args) for s, (pool, args) in enumerate(zip(_tp_pools(), args_per_gpu))]
    results, errors = [], []
    for future in futures:
        try:
            results.append(future.result())
        except BaseException as e:
            errors.append(e)
    if errors:  # the real error, not the other thread's broken barrier
        raise next((e for e in errors if not isinstance(e, threading.BrokenBarrierError)), errors[0])
    return results


def _exchange(s, hidden, partial, gate, segments, shared, sliced):
    """hidden += gate * (partial_0 + partial_1) on GPU s, from GPU s's thread. Both copies are queued (each on its source
    GPU's stream) before either thread adds into its own partial sum, so no copy can read a sum that was already changed."""
    other = 1 - s
    shared[s] = partial
    _TP_BARRIER.wait()  # both partial sums are queued
    for a, b in (_row_chunks(hidden.shape[0]) if sliced else [(0, hidden.shape[0])]):
        incoming = shared[other][a:b].to(DEVICES[s], non_blocking=True, copy=True)
        _TP_BARRIER.wait()  # both copies of these rows are queued
        partial[a:b].add_(incoming)
        del incoming
    shared[s] = None  # the other thread has queued its last read of it (before the barrier above)
    return _gated_residual([hidden], [gate], [partial], segments)


def _half_block(s, block, hidden, temb, rope, segments, shared, sliced):
    """GPU s's half of a DiT block, queued from GPU s's thread: the same kernels as TPBlock's one-thread path."""
    shard = block.shards[s]
    mod = shard.adaln_proj(temb)
    if sliced:
        partial = _sliced_attention(shard, hidden, mod, segments, rope)
    else:  # the modulated input is handed over in a list: attention frees it once Q, K and V exist
        partial = shard.attn([_norm_modulate(shard.norm1, hidden, [mod[0], mod[1]], segments)], rope=rope)
    hidden = _exchange(s, hidden, partial, mod[2], segments, shared, sliced)
    del partial
    if sliced:
        partial = _sliced_mlp(shard, hidden, mod, segments)
    else:
        partial = shard.mlp([_norm_modulate(shard.norm2, hidden, [mod[3], mod[4]], segments)])
    return _exchange(s, hidden, partial, mod[5], segments, shared, sliced)


class TPBlock(torch.nn.Module):
    def __init__(self, block, index=0):
        super().__init__()
        self.index = index
        attn, mlp = block.attn, block.mlp
        heads, head_dim = attn.heads, attn.head_dim
        inner, hidden = heads * head_dim, mlp.fc2.out_features
        ffn = mlp.fc2.in_features
        half_inner, half_ffn = inner // 2, ffn // 2
        if getattr(attn, "qkv_interleaved", False):
            raise RuntimeError("The tensor-parallel split expects grouped Q|K|V rows (quantized H3 checkpoints)")
        self.splits = []
        shards = []
        for s, dev in enumerate(DEVICES):
            h = torch.arange(s * half_inner, (s + 1) * half_inner)
            f = torch.arange(s * half_ffn, (s + 1) * half_ffn)
            qkv_rows = torch.cat((h, h + inner, h + 2 * inner))
            fc1_rows = torch.cat((f, f + ffn))  # SwiGLU: gate rows, then value rows
            out_cols = slice(s * half_inner, (s + 1) * half_inner)
            fc2_cols = slice(s * half_ffn, (s + 1) * half_ffn)
            self.splits.append({"qkv": qkv_rows, "fc1": fc1_rows, "out": out_cols, "fc2": fc2_cols})
            with torch.device("meta"):
                sattn = H3Attention(hidden, heads // 2, head_dim, attn.q_norm.eps, dtype=torch.bfloat16)
                smlp = H3MLP(hidden, half_ffn, dtype=torch.bfloat16)
            sattn.qkv_proj = Int8Shard(attn.qkv_proj, dev, rows=qkv_rows)
            sattn.out_proj = Int8Shard(attn.out_proj, dev, cols=out_cols)
            sattn.q_norm = copy.deepcopy(attn.q_norm).to(dev)
            sattn.k_norm = copy.deepcopy(attn.k_norm).to(dev)
            smlp.fc1 = Int8Shard(mlp.fc1, dev, rows=fc1_rows)
            smlp.fc2 = Int8Shard(mlp.fc2, dev, cols=fc2_cols)
            shard = torch.nn.Module()
            shard.attn, shard.mlp = sattn, smlp
            shard.norm1 = copy.deepcopy(block.norm1).to(dev)
            shard.norm2 = copy.deepcopy(block.norm2).to(dev)
            shard.adaln_proj = copy.deepcopy(block.adaln_proj).to(dev)
            shards.append(shard)
        self.shards = torch.nn.ModuleList(shards)

    @property
    def attn(self):  # H3's forward checks block.attn for q_proj / vdn
        return self.shards[1].attn

    def forward(self, x_list, temb, segments, rope, residual_signature_elements=0):
        if residual_signature_elements:
            raise RuntimeError("First Block Cache is not supported by the tensor-parallel DiT")
        hidden = _take(x_list)
        hidden = hidden if hidden.device == MAIN_DEV else hidden.to(MAIN_DEV)
        replica = _tp_cache["replica"] if _tp_cache["src"] is hidden else hidden.to(DIT_DEV, copy=True)
        hiddens = [replica, hidden]
        tembs = [_copy_to(temb, DIT_DEV, "temb"), temb]
        ropes = [_copy_to(rope, DIT_DEV, "rope"), rope]
        TP_STATE["block"] = self.index
        if TP_THREADS["on"] and TP_THREADS["warm"]:
            shared = [None] * len(DEVICES)
            hiddens = _run_on_both(_half_block, [(self, hiddens[s], tembs[s], ropes[s], segments, shared, LONG_VIDEO["on"])
                                                 for s in range(len(DEVICES))])
            _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
            return hiddens[1]
        if LONG_VIDEO["on"]:
            return self._forward_sliced(hiddens, tembs, ropes, segments)
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            shard = self.shards[s]
            with _on(dev):
                mod = shard.adaln_proj(tembs[s])
                mods.append(mod)
                h = _norm_modulate(shard.norm1, hiddens[s], [mod[0], mod[1]], segments)
                partials.append(shard.attn([h], rope=ropes[s]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            shard = self.shards[s]
            with _on(dev):
                h = _norm_modulate(shard.norm2, hiddens[s], [mods[s][3], mods[s][4]], segments)
                partials.append(shard.mlp([h]))
        hiddens = _all_reduce_residual(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]

    def _forward_sliced(self, hiddens, tembs, ropes, segments):
        mods, partials = [], []
        for s, dev in enumerate(DEVICES):  # kernels are queued on both GPUs before either is waited for
            with _on(dev):
                mods.append(self.shards[s].adaln_proj(tembs[s]))
                partials.append(_sliced_attention(self.shards[s], hiddens[s], mods[s], segments, ropes[s]))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[2] for m in mods], segments)
        partials = []
        for s, dev in enumerate(DEVICES):
            with _on(dev):
                partials.append(_sliced_mlp(self.shards[s], hiddens[s], mods[s], segments))
        hiddens = _all_reduce_sliced(hiddens, partials, [m[5] for m in mods], segments)
        _tp_cache["src"], _tp_cache["replica"] = hiddens[1], hiddens[0]
        return hiddens[1]


def split_transformer(model, progress=None):
    """Every DiT block becomes a TPBlock; the rest of the transformer (embeddings, final layer) moves to GPU 1."""
    for index in range(len(model.blocks)):
        model.blocks[index] = TPBlock(model.blocks[index], index)
        if progress is not None:
            progress(index + 1)
    for name, child in model.named_children():
        if name not in ("blocks", "token_refiner"):
            child.to(MAIN_DEV)
    for name, buffer in list(model.named_buffers(recurse=False)):
        model._buffers[name] = buffer.to(MAIN_DEV)
# <<< section


NUM_BLOCKS = len(transformer.blocks)
BAR.start("split", "Loading: splitting the DiT", NUM_BLOCKS, "blocks")
split_transformer(transformer, BAR.update)
BAR.finish()
gc.collect()

# ==== Exact-size pinned RAM for the token refiner ====
# The 1.4 GiB token refiner runs once per prompt: keep it in page-locked RAM (cudaHostRegister: exactly the bytes asked
# for, unlike PyTorch's power-of-two pinned blocks) and swap it onto GPU 1 only for that.
_HOST_REGISTERED = []  # registered RAM stays alive (and pinned) for the whole session
PINNED_BYTES = {}


def _dont_fork(ptr, nbytes):
    """madvise(MADV_DONTFORK): a fork() must not duplicate pinned RAM (Linux copies pinned pages into the child at once)."""
    try:
        import ctypes
        page = os.sysconf("SC_PAGE_SIZE")
        start = -(-ptr // page) * page
        end = (ptr + nbytes) // page * page
        if end > start:
            ctypes.CDLL("libc.so.6").madvise(ctypes.c_void_p(start), ctypes.c_size_t(end - start), 10)  # MADV_DONTFORK
    except Exception:
        pass


def pinned_empty_exact(nbytes):
    t = torch.empty(nbytes, dtype=torch.uint8)
    torch.cuda.check_error(torch.cuda.cudart().cudaHostRegister(t.data_ptr(), nbytes, 0))
    _dont_fork(t.data_ptr(), nbytes)
    _HOST_REGISTERED.append(t)
    return t


_refiner = transformer.token_refiner
_ref_list = list(_refiner.parameters())
_ref_offsets, _ref_total = [], 0
for _p in _ref_list:
    _ref_offsets.append(_ref_total)
    _ref_total += -(-_p.numel() * _p.element_size() // 64) * 64
try:
    _ref_buf = pinned_empty_exact(_ref_total)
    _refiner_params = []
    for _p, _off in zip(_ref_list, _ref_offsets):
        _cpu = _ref_buf[_off:_off + _p.numel() * _p.element_size()].view(_p.dtype).view(_p.shape)
        _cpu.copy_(_p.data)
        _refiner_params.append((_p, _cpu))
except Exception as e:  # no cudaHostRegister: PyTorch's (rounded) pinned blocks
    print(f"⚠️ Exact pinning unavailable ({e}); using PyTorch pinned memory")
    _refiner_params = [(p, p.data.to("cpu").pin_memory()) for p in _ref_list]
PINNED_BYTES["token refiner"] = _ref_total
for _p, _cpu in _refiner_params:
    _p.data = _cpu
_orig_preprocess_text = transformer.preprocess_text_embeds


def _preprocess_text_embeds(text_states):
    if text_states.shape[-1] == transformer.hidden_size:
        return text_states
    for p, cpu in _refiner_params:
        p.data = cpu.to(MAIN_DEV, non_blocking=True)
    try:
        return _orig_preprocess_text(text_states.to(MAIN_DEV))
    finally:
        for p, cpu in _refiner_params:
            p.data = cpu


transformer.preprocess_text_embeds = _preprocess_text_embeds

# ==== Text encoder + VAEs: MMGP on GPU 1, unloaded before denoising ====
pipeline.audio_vae._convertWeightsFloatTo = None  # keep the audio VAE in FP32 (Turing has no BF16 convs)
aux_pipe = {
    "text_encoder": pipeline.text_encoder.language_model,
    "vision_encoder": pipeline.text_encoder.visual,
    "vae": pipeline.video_decoder,
    "video_encoder": pipeline.video_encoder,
    "audio_vae": pipeline.audio_vae,
}
pipeline.video_decoder._budget = 0  # decode with the whole VAE resident (it is called once per clip and tile)
if TE_DTYPE == torch.float32:
    pipeline.text_encoder.language_model._convertWeightsFloatTo = None
    pipeline.text_encoder.visual._convertWeightsFloatTo = None
# The text encoder (~14 GB) and the video VAE decoder (2.3 GB) are pinned in RAM: Kaggle's disk reads at ~180 MB/s, so
# a memory-mapped copy that dropped out of the page cache would cost ~15 s per video before decoding could start.
_pin = ["text_encoder", "vae", "video_encoder", "audio_vae"]  # +1 GB of RAM: the encoder and audio VAE are not re-read from disk per video
_profile_args = dict(profile_no=4, quantizeTransformer=False, extraModelsToQuantize=None, convertWeightsFloatTo=torch.bfloat16,
                     asyncTransfers=True, verboseLevel=-1, budgets={"text_encoder": 1500, "vision_encoder": 0, "*": 3000})
BAR.start("pin", "Loading: pinning RAM")
try:
    aux_offload = offload.profile(aux_pipe, pinnedMemory=_pin, **_profile_args)
except Exception as e:
    print(f"⚠️ Pinning failed ({e}); the text encoder streams from regular RAM")
    aux_offload = offload.profile(aux_pipe, pinnedMemory=False, **_profile_args)
pipeline.set_offload_handoff(aux_offload, None)  # the pipeline unloads these before handing the GPU to the DiT
BAR.finish()
if hasattr(torch._C, "_host_emptyCache"):
    torch._C._host_emptyCache()

gc.collect()
for _dev in DEVICES:
    with torch.cuda.device(_dev):
        torch.cuda.empty_cache()
_free0, _free1 = torch.cuda.mem_get_info(0)[0], torch.cuda.mem_get_info(1)[0]


# ==== Attention: exact FP16 SDPA, or Comfy Kitchen's INT8 attention ====
# Attention is most of a denoising step (the source video is part of the sequence: ~31k tokens at 480p 5.2 s).
# Fast (default): Comfy Kitchen's INT8 attention, a compiled CUDA kernel with Turing (SM 7.5) code. Q and K get a Hadamard
# rotation (same dot products, smaller outliers) and go to INT8, V to INT8 per channel, the softmax weights to unsigned
# INT8, with the softmax itself in FP32; the T4's INT8 tensor cores run at twice the FP16 rate. On 2 x T4 it halves a
# 480p step (about 100 s to 50 s) and stays within ~1 % of exact attention. It is offered only when a startup test on
# both GPUs passes and shows it faster, and every INT8 run still compares it with exact attention on the user's video in
# its first step (blocks 1 and 26, GPU 0), switching back to exact by itself if they differ too much.
# Exact: PyTorch SDPA in FP16 (its memory-efficient kernel; the T4 has no flash kernel).
# Wan2GP's own SageAttention path is Triton INT8 code, which Kaggle's Triton cannot compile for the T4.
# >>> section:attention
ATTENTION_EXACT, ATTENTION_INT8 = "Exact (FP16)", "Fast (Kitchen INT8)"
ATTENTION_MAX_DIFF = 0.08  # relative L2 difference from exact attention above which an INT8 run switches back to exact
INT8_MIN_TOKENS = 2048     # INT8 only for the DiT's long sequences (the text token refiner stays exact)
INT8_DEVICE_TYPES = ("cuda",)
ATTENTION = {"mode": "sdpa", "int8": None, "int8_dtype": None, "int8_contiguous": False, "int8_speedup": None, "int8_reason": "",
             "checks": set(), "fallback": None}
offload.shared_state["_attention"] = "sdpa"
import shared.attention as _w2g_attention
_sdpa_attention = _w2g_attention.pay_attention  # Wan2GP's own (SDPA here), never the wrapper below
_ATTENTION_CHECK_LOCK = threading.Lock()


def _int8_attention(q, k, v):
    """Kitchen INT8 attention on [batch, tokens, heads, dim] tensors (H3's layout), output in q's dtype and layout."""
    out_dtype, dtype = q.dtype, ATTENTION["int8_dtype"] or q.dtype
    args = [t.transpose(1, 2) if t.dtype == dtype else t.transpose(1, 2).to(dtype) for t in (q, k, v)]
    if ATTENTION["int8_contiguous"]:
        args = [t.contiguous() for t in args]
    return ATTENTION["int8"].int8_attention(*args).transpose(1, 2).to(out_dtype)


def _relative_diff(out, ref):
    """||out - ref|| / ||ref|| in FP32, by slices of tokens (no full-size FP32 copies)."""
    num = torch.zeros((), dtype=torch.float32, device=out.device)
    den = torch.zeros((), dtype=torch.float32, device=out.device)
    for a in range(0, out.shape[1], 4096):
        o, r = out[:, a:a + 4096].float(), ref[:, a:a + 4096].float()
        num += (o - r).square().sum()
        den += r.square().sum()
    return (num / den.clamp_min(1e-30)).sqrt().item()


def _check_int8_attention(q, k, v):
    """INT8 vs exact attention on this block's real Q / K / V. Returns the output to use (exact if INT8 is too far off)."""
    dev = q.device
    if dev.type == "cuda":
        free = torch.cuda.mem_get_info(dev)[0] + torch.cuda.memory_reserved(dev) - torch.cuda.memory_allocated(dev)
        if free < 8 * q.numel() * q.element_size() + (512 << 20):  # the exact reference needs copies of Q / K / V
            return _int8_attention(q, k, v)
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    out = _int8_attention(q, k, v)
    diff = _relative_diff(out, ref)
    if not math.isfinite(diff) or diff > ATTENTION_MAX_DIFF:
        ATTENTION.update(mode="sdpa", fallback=diff)
        return ref
    return out


def pay_attention_tp(qkv_list, recycle_q=False, **kwargs):
    """H3's attention call: Kitchen INT8 for the DiT's long sequences when the run asks for it, else Wan2GP's (SDPA)."""
    q = qkv_list[0]
    if (ATTENTION["mode"] != "int8" or kwargs or q.dim() != 4 or q.shape[1] < INT8_MIN_TOKENS
            or q.device.type not in INT8_DEVICE_TYPES):
        return _sdpa_attention(qkv_list, recycle_q=recycle_q, **kwargs)
    q, k, v = qkv_list
    qkv_list.clear()
    check = False
    if q.device == DEVICES[0] and ATTENTION["checks"]:
        with _ATTENTION_CHECK_LOCK:  # one check per listed block, whichever thread gets there first
            check = TP_STATE["block"] in ATTENTION["checks"]
            ATTENTION["checks"].discard(TP_STATE["block"])
    return _check_int8_attention(q, k, v) if check else _int8_attention(q, k, v)


_h3_transformer.pay_attention = pay_attention_tp


def _probe_case(dev, fused):
    """INT8 vs SDPA difference on random BF16 data in one of H3's two layouts: fused (Q and K are views of one fused
    projection output, V a copy: the normal path) or separate contiguous tensors (long-video mode)."""
    g = torch.Generator(device=dev).manual_seed(0)
    tokens, heads, dim = 3000, 4, 128
    if fused:
        qkv = torch.randn(1, tokens, 3 * heads * dim, device=dev, generator=g).to(torch.bfloat16)
        q, k, v = (t.view(1, tokens, heads, dim) for t in qkv.split(heads * dim, dim=-1))
        v = v.clone()
    else:
        q, k, v = (torch.randn(1, tokens, heads, dim, device=dev, generator=g).to(torch.bfloat16) for _ in range(3))
    ref = _sdpa_attention([q.clone(), k.clone(), v.clone()])
    return _relative_diff(_int8_attention(q, k, v), ref)


def _probe_int8_attention():
    """Kitchen INT8 attention on both GPUs against SDPA on random data (catches a kernel that does not load, a dtype it
    does not take on Turing, or strides it does not follow), then a speed test at a DiT-like size."""
    try:
        import comfy_kitchen
        if not all(comfy_kitchen.int8_attention_is_available(dev) for dev in DEVICES):
            ATTENTION["int8_reason"] = "not available on this GPU"
            return
        ATTENTION["int8"] = comfy_kitchen
        # BF16 straight from the projections and the projections' own strides first; FP16 / contiguous copies otherwise
        for dtype, contiguous in ((torch.bfloat16, False), (torch.bfloat16, True), (torch.float16, False), (torch.float16, True)):
            ATTENTION.update(int8_dtype=dtype, int8_contiguous=contiguous)
            try:
                worst = 0.0
                for dev in DEVICES:
                    with _on(dev), torch.inference_mode():
                        worst = max(worst, _probe_case(dev, True), _probe_case(dev, False))
                if math.isfinite(worst) and worst < 0.1:
                    break
                ATTENTION["int8_reason"] = f"self-test differed {worst:.1%} from SDPA"
            except Exception as e:
                ATTENTION["int8_reason"] = f"self-test failed: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}"
            ATTENTION["int8_dtype"] = None
        if ATTENTION["int8_dtype"] is None:
            ATTENTION["int8"] = None
            return
        ATTENTION["int8_reason"] = ""
        with _on(MAIN_DEV), torch.inference_mode():
            q, k, v = (torch.randn(1, 8192, 28, 128, device=MAIN_DEV).to(torch.bfloat16) for _ in range(3))
            timings = []
            for run in (lambda: _sdpa_attention([q, k, v]), lambda: _int8_attention(q, k, v)):
                for _ in range(2):
                    run()
                start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(3):
                    run()
                end.record()
                end.synchronize()
                timings.append(start.elapsed_time(end))
            ratio = timings[0] / max(timings[1], 1e-3)
            del q, k, v
        if ratio < 1.05:  # offered only when it is actually faster on these GPUs
            ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"not faster than exact attention here: {ratio:.2f}x")
        else:
            ATTENTION["int8_speedup"] = ratio
    except Exception as e:
        ATTENTION.update(int8=None, int8_dtype=None, int8_reason=f"unavailable: {str(e).splitlines()[0][:120] if str(e) else type(e).__name__}")
    finally:
        for dev in DEVICES:
            with _on(dev):
                torch.cuda.empty_cache()


def attention_speedup(choice):
    return (ATTENTION["int8_speedup"] or 1.0) if choice == ATTENTION_INT8 and ATTENTION["int8"] is not None else 1.0
# <<< section


BAR.start("attention", "Loading: testing Kitchen INT8 attention", 1, "test")
_probe_int8_attention()
BAR.finish()
ATTENTION_CHOICES = ([ATTENTION_INT8] if ATTENTION["int8"] is not None else []) + [ATTENTION_EXACT]
ATTENTION_DEFAULT = ATTENTION_CHOICES[0]
ATTENTION_LABEL = "Kitchen INT8 attention" if ATTENTION["int8"] is not None else "SDPA attention"

# ==== VAE encode and decode on both GPUs ====
# The video VAE encodes and decodes in independent temporal clips (only the decoder's cross-fade between neighbours is
# sequential), so the clips are split between the two GPUs: GPU 1 uses the MMGP-managed VAE, GPU 0 a second copy whose
# weights are copied from the first one only for the duration of the encode or decode (no second copy in RAM; between
# videos it holds no weights at all). Tiling, attention and cross-fades are exactly Wan2GP's. On the first video GPU 0
# also processes clip 0 and must match GPU 1's result; otherwise that direction falls back to GPU 1 alone.
_vae_b = h3_main._load_video_vae(VIDEO_VAE_INT8_FILE, False)  # same architecture; its own weights are dropped below
_VAE_B_GROUPS = {  # copy on GPU 0 -> the MMGP-managed modules it mirrors
    "decode": (torch.nn.ModuleDict({"post_quant_conv": _vae_b.post_quant_conv, "decoder": _vae_b.decoder}), pipeline.video_decoder),
    "encode": (torch.nn.ModuleDict({"encoder": _vae_b.encoder, "quant_conv": _vae_b.quant_conv}), pipeline.video_encoder),
}
_vae_b_slots = {}  # group -> [(module, "_parameters" | "_buffers", name, source module)]
for _group, (_parts, _source) in _VAE_B_GROUPS.items():
    _source_modules = dict(_source.named_modules())
    _vae_b_slots[_group] = [(_module, _store, _key, _source_modules[_name])
                            for _name, _module in _parts.named_modules() for _store in ("_parameters", "_buffers")
                            for _key, _value in getattr(_module, _store).items() if _value is not None]


def _vae_b_attach(device, group="decode"):
    """Copy the first VAE's weights of this group (pinned RAM, or GPU 1 if MMGP has them loaded) to `device`."""
    for module, store, key, source in _vae_b_slots[group]:
        value = getattr(source, store)[key].to(device, non_blocking=True)
        if store == "_parameters":
            value = torch.nn.Parameter(value, requires_grad=False)
        getattr(module, store)[key] = value


def _vae_b_release(group=None):
    """Drop the copy's weights. It must not keep references to the first VAE's tensors either: MMGP swaps in new GPU
    tensors when it loads a model, so a reference taken while they are on GPU 1 would keep that VRAM alive after MMGP
    unloads it."""
    for name in ([group] if group else list(_vae_b_slots)):
        for module, store, key, _ in _vae_b_slots[name]:
            getattr(module, store)[key] = None


_vae_b_release()
gc.collect()

# Wan2GP decodes each clip as 256 px tiles stacked into ONE decoder batch (~12 tiles at 480p, ~28 at 720p). The
# decoder treats every tile on its own, so running the tiles in smaller groups gives exactly the same frames. The group
# size is planned per GPU from its free VRAM before each decode; halving on an out-of-memory error stays as a last resort.
VAE_TILE_GROUP = {}
VAE_TILE_BYTES = 100 * 2**20


def _tile_grouped(decoder):
    full_forward = decoder.forward

    def forward(hidden_states, *args, **kwargs):
        dev, tiles = hidden_states.device.index, hidden_states.shape[0]
        while True:
            n = min(VAE_TILE_GROUP.get(dev, tiles), tiles)
            try:
                if n >= tiles:
                    return full_forward(hidden_states, *args, **kwargs)
                return torch.cat([full_forward(hidden_states[i:i + n], *args, **kwargs) for i in range(0, tiles, n)])
            except torch.OutOfMemoryError:
                if n <= 1:
                    raise
                VAE_TILE_GROUP[dev] = max(1, n // 2)
                print(f"  ⚙️ VAE: GPU {dev} decodes {VAE_TILE_GROUP[dev]} tiles at a time from now on (out of memory with {n})")
                with torch.cuda.device(hidden_states.device):
                    torch.cuda.synchronize()  # kernels queued before the failed allocation may still use freed blocks
                    gc.collect()
                    torch.cuda.empty_cache()

    decoder.forward = forward


_tile_grouped(pipeline.vae.decoder)
_tile_grouped(_vae_b.decoder)


def _plan_tile_groups():
    """Tiles per decoder call that fit each GPU's free VRAM right now (GPU 1 loads its 2.3 GB decoder on first use)."""
    resident = any(p.is_cuda for p in pipeline.video_decoder.parameters())
    decoder_bytes = sum(p.numel() * p.element_size() for p in pipeline.video_decoder.parameters())
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info(dev)[0]
        if dev == MAIN_DEV and not resident:
            free -= decoder_bytes
        VAE_TILE_GROUP[dev.index] = max(1, int((free - 512 * 2**20) // VAE_TILE_BYTES))


# ==== Drop the checkpoint files from the page cache ====
# Loading reads ~40 GB of checkpoints and Linux keeps what fits of them cached. That cache is reclaimable, but Kaggle's
# RAM meter counts it. Everything is pinned or on the GPUs now, so the cached copies are useless.
def drop_file_cache(*roots):
    for root in roots:
        for dirpath, _, files in os.walk(root, followlinks=True):
            for name in files:
                try:
                    fd = os.open(os.path.join(dirpath, name), os.O_RDONLY)
                    try:
                        os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                    finally:
                        os.close(fd)
                except (OSError, AttributeError):
                    pass


def _ram_report():
    vm = psutil.virtual_memory()
    cache = getattr(vm, "cached", 0) + getattr(vm, "buffers", 0)
    return f"{_ram_in_use()} used by the engine | {_gb(cache)} file cache (reclaimable, Kaggle's meter counts it)"


drop_file_cache(MODELS_DIR)
_orig_vae_decode = pipeline.vae._decode
PARALLEL_VAE = {"enabled": True, "verified": False}
PEAKS = {}


# >>> section:venc
PARALLEL_ENC = {"enabled": True, "verified": False, "scale": 1}


def parallel_vae_encode(x, device=None):
    """Wan2GP's _encode (clip_length-frame clips, the last one padded with its last frame, token_drop latents dropped at
    the end) with the clips split between the two GPUs. Returns the moments on GPU 1, like the original."""
    vae = pipeline.vae
    clip_length = vae.config.clip_length
    starts = list(range(0, x.shape[2], clip_length))
    if len(starts) < 2 or not PARALLEL_ENC["enabled"]:
        return _orig_vae_encode(x, device)

    def clip_at(start):
        clip = x[:, :, start:start + clip_length]
        if clip.shape[2] < clip_length:
            clip = torch.cat([clip, x[:, :, -1:].repeat(1, 1, clip_length - clip.shape[2], 1, 1)], dim=2)
        return clip

    _vae_b.use_tiling = vae.use_tiling
    for attr in ("tile_sample_min_height", "tile_sample_min_width", "tile_sample_min_overlap_height", "tile_sample_min_overlap_width"):
        setattr(_vae_b, attr, getattr(vae, attr))
    moments, check, errors = [None] * len(starts), {}, []

    def work(dev, model, indices):
        try:
            with _on(dev), torch.inference_mode():
                for i in indices:
                    if pipeline._interrupt:
                        return
                    out = model._encode_clip(clip_at(starts[max(i, 0)]), dev).to("cpu")
                    if i < 0:  # verification copy of clip 0 on GPU 0
                        check["clip"] = out
                    else:
                        moments[i] = out
        except BaseException as e:
            errors.append(e)

    with _on(DIT_DEV):
        _vae_b_attach(DIT_DEV, "encode")
    gpu0 = list(range(1, len(starts), 2))
    if not PARALLEL_ENC["verified"]:
        gpu0 = [-1] + gpu0
    PARALLEL_ENC["scale"] = 2  # the encode progress counts GPU 1's tiles only
    try:
        helper = threading.Thread(target=work, args=(DIT_DEV, _vae_b, gpu0))
        helper.start()
        work(MAIN_DEV, vae, list(range(0, len(starts), 2)))
        helper.join()
    finally:
        PARALLEL_ENC["scale"] = 1
        try:
            for dev in DEVICES:
                if dev.type == "cuda":
                    torch.cuda.synchronize(dev)
        except Exception as e:
            errors.append(e)
        _vae_b_release("encode")
    if errors:
        raise errors[0]
    if pipeline._interrupt:
        from models.minimax_h3.interrupt import GenerationInterrupted
        raise GenerationInterrupted
    if not PARALLEL_ENC["verified"] and "clip" in check:
        a, b = check["clip"].float(), moments[0].float()
        mismatch = ((a - b).norm() / b.norm().clamp_min(1e-6)).item()
        PARALLEL_ENC["verified"] = True
        if mismatch > 1e-2:
            PARALLEL_ENC["enabled"] = False
            print(f"⚠️ GPU 0's VAE encoder disagrees with GPU 1 (rel diff {mismatch:.3f}); encoding on GPU 1 only from now on")
            with _on(MAIN_DEV), torch.inference_mode():
                for i in range(1, len(starts), 2):
                    moments[i] = vae._encode_clip(clip_at(starts[i]), MAIN_DEV).to("cpu")
    out = torch.cat(moments, dim=2)
    if vae.config.token_drop > 0:
        out = out[:, :, :-vae.config.token_drop]
    return out.to(MAIN_DEV)
# <<< section


_orig_vae_encode = pipeline.vae._encode
pipeline.vae._encode = parallel_vae_encode


# >>> section:vae
def parallel_vae_decode(z, uint8_rounding=None):
    vae = pipeline.vae
    chunk_tokens, token_drop = vae.tokens_chunk_size, vae.config.token_drop
    ratio = vae.temporal_compression_ratio
    chunk_frames = chunk_tokens * ratio
    if vae.decoder.patch_size_t == 1 or z.shape[2] == 1:
        return _orig_vae_decode(z, uint8_rounding)
    num_tokens = z.shape[2] + token_drop
    pad_tokens = (-num_tokens) % chunk_tokens
    num_chunks = (num_tokens + pad_tokens) // chunk_tokens - int(token_drop > 0)
    if num_chunks < 2 or not PARALLEL_VAE["enabled"]:
        return _orig_vae_decode(z, uint8_rounding)
    if pad_tokens > 0:
        z = torch.cat([z, z[:, :, -1:].repeat(1, 1, pad_tokens, 1, 1)], dim=2)
    intra_tail = vae.config.clip_length % ratio
    tokens_before_pad = z.shape[2] - pad_tokens
    pad_frames = sum(intra_tail if intra_tail and (tokens_before_pad + i) % chunk_tokens == 0 else ratio
                     for i in range(pad_tokens))
    output_frames = num_chunks * (chunk_frames - vae.frame_pre_padding) + vae.frame_overlap - pad_frames

    _vae_b.use_tiling = vae.use_tiling
    for attr in ("tile_sample_min_height", "tile_sample_min_width", "tile_sample_min_overlap_height", "tile_sample_min_overlap_width"):
        setattr(_vae_b, attr, getattr(vae, attr))
    PEAKS["denoise"] = [torch.cuda.max_memory_allocated(d) for d in DEVICES]  # the DiT phase ends here
    for d in DEVICES:
        torch.cuda.reset_peak_memory_stats(d)
    clips, check, errors, done, lock = [None] * num_chunks, {}, [], [0], threading.Lock()
    BAR.start("decode", "VAE decoding", num_chunks, "clips")

    def decode_one(dev, model, i):
        start = i * chunk_tokens
        clip = model._decode_clip(z[:, :, start:start + chunk_tokens + vae.token_overlap].to(dev))
        return clip.to("cpu")

    def work(dev, model, indices):
        try:
            with torch.cuda.device(dev), torch.inference_mode():
                for i in indices:
                    if pipeline._interrupt:
                        return
                    if i < 0:  # verification copy of clip 0 on GPU 0
                        check["clip"] = decode_one(dev, model, 0)
                        continue
                    clips[i] = decode_one(dev, model, i)
                    with lock:
                        done[0] += 1
                        BAR.update(done[0])
                        report(0.88 + 0.08 * done[0] / num_chunks, f"🎞️ VAE decoding {done[0]}/{num_chunks} clips")
        except BaseException as e:
            errors.append(e)

    with torch.cuda.device(DIT_DEV):
        torch.cuda.empty_cache()
        _vae_b_attach(DIT_DEV, "decode")
    _plan_tile_groups()
    gpu0_clips = list(range(1, num_chunks, 2))
    if not PARALLEL_VAE["verified"]:
        gpu0_clips = [-1] + gpu0_clips
    try:
        helper = threading.Thread(target=work, args=(DIT_DEV, _vae_b, gpu0_clips))
        helper.start()
        work(MAIN_DEV, vae, list(range(0, num_chunks, 2)))
        helper.join()
    finally:
        try:  # let every queued kernel finish before GPU 0's decoder weights are dropped and its memory released
            for dev in DEVICES:
                torch.cuda.synchronize(dev)
        except Exception as e:
            errors.append(e)
        _vae_b_release("decode")  # drop GPU 0's copy
        if not errors:
            with torch.cuda.device(DIT_DEV):
                torch.cuda.empty_cache()
    if errors:
        raise errors[0]
    if pipeline._interrupt:
        from models.minimax_h3.interrupt import GenerationInterrupted
        raise GenerationInterrupted
    if not PARALLEL_VAE["verified"] and "clip" in check:
        a, b = check["clip"].float(), clips[0].float()
        mismatch = ((a - b).norm() / b.norm().clamp_min(1e-6)).item()
        PARALLEL_VAE["verified"] = True
        if mismatch > 1e-2:
            PARALLEL_VAE["enabled"] = False
            print(f"⚠️ GPU 0's VAE copy disagrees with GPU 1 (rel diff {mismatch:.3f}); decoding on GPU 1 only from now on")
            with torch.cuda.device(MAIN_DEV), torch.inference_mode():
                for i in range(1, num_chunks, 2):
                    clips[i] = decode_one(MAIN_DEV, vae, i)
    return stitch_clips(vae, clips, chunk_frames, token_drop, output_frames, uint8_rounding)


def stitch_clips(vae, clips, chunk_frames, token_drop, output_frames, uint8_rounding):
    """Wan2GP's _decode loop over already decoded clips: cross-fades, then conversion (uint8 when asked) into one CPU video."""
    BAR.start("stitch", "Stitching clips")
    decoded, write_position, overlap = None, 0, None

    def write_chunk(chunk):
        nonlocal decoded, write_position
        copy_frames = min(chunk.shape[2], output_frames - write_position)
        for start in range(0, copy_frames, 4):
            part = vae._prepare_decoded_chunk(chunk[:, :, start:min(start + 4, copy_frames)], uint8_rounding)
            if decoded is None:
                decoded = torch.empty(*part.shape[:2], output_frames, *part.shape[3:], dtype=part.dtype, device="cpu")
            decoded[:, :, write_position:write_position + part.shape[2]].copy_(part)
            write_position += part.shape[2]

    for i in range(len(clips)):
        clip = clips[i]
        clips[i] = None
        for j in range(int(token_drop > 0) + 1):
            chunk = clip[:, :, j * chunk_frames:(j + 1) * chunk_frames][:, :, vae.frame_pre_padding:]
            if j == 0:
                if overlap is not None:
                    chunk = vae._blend(overlap, chunk, vae.frame_overlap, dim=-3)
                write_chunk(chunk)
            else:
                overlap = chunk.contiguous()
    if overlap is not None:
        write_chunk(overlap)
    if write_position != output_frames:
        raise RuntimeError(f"MiniMax H3 VAE decoded {write_position} frames, expected {output_frames}")
    return decoded
# <<< section


pipeline.vae._decode = parallel_vae_decode
_orig_audio_decode = pipeline.audio_vae.decode


def _audio_decode(*args, **kwargs):
    BAR.start("audio", "Audio decoding")
    return _orig_audio_decode(*args, **kwargs)


pipeline.audio_vae.decode = _audio_decode


# >>> section:io
# Viggle Remake windows are 17n + 5 frames at 24 fps, 90 to 345 frames.
FPS = 24
WINDOW_FRAMES = [n for n in range(VIGGLE_REMAKE_MIN_FRAMES, VIGGLE_REMAKE_MAX_FRAMES + 1) if (n - 5) % 17 == 0]
DURATIONS = {f"{n / FPS:.1f} s · {n} frames": n for n in WINDOW_FRAMES}
DEFAULT_DURATION = next(k for k, v in DURATIONS.items() if v == 124)
RESOLUTIONS = {"480p (832×480 class) ⚡ recommended": 480, "576p (1024×576 class)": 576, "720p (1280×736 class) 🐢 slow": 736}
DEFAULT_RESOLUTION = next(iter(RESOLUTIONS))
MAX_PAINTED = 4


def probe_video(path):
    """Width, height (after rotation), frame rate, duration in seconds, and whether it has an audio stream."""
    out = subprocess.run(["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", path],
                         capture_output=True, text=True, check=True).stdout
    info = json.loads(out)
    video = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)
    if video is None:
        raise ValueError("This file has no video stream.")
    width, height = int(video["width"]), int(video["height"])
    rotation = 0
    for side in video.get("side_data_list", []) or []:
        if "rotation" in side:
            rotation = int(float(side["rotation"]))
    rotation = int(float((video.get("tags") or {}).get("rotate", rotation)))
    if rotation % 180:
        width, height = height, width
    num, _, den = (video.get("avg_frame_rate") or video.get("r_frame_rate") or "24/1").partition("/")
    fps = float(num) / float(den or 1) if float(num or 0) else 24.0
    duration = float(video.get("duration") or info.get("format", {}).get("duration") or 0.0)
    has_audio = any(s.get("codec_type") == "audio" for s in info.get("streams", []))
    return {"width": width, "height": height, "fps": fps, "duration": duration, "audio": has_audio}


def canvas_size(short_edge, width, height):
    """The generation size: the source's proportions with the given short edge, both sides multiples of 32 (H3's latent
    patches), like the official sampler's --short-edge."""
    if width >= height:
        h = short_edge
        w = max(256, round(short_edge * width / height / 32) * 32)
    else:
        w = short_edge
        h = max(256, round(short_edge * height / width / 32) * 32)
    return int(w), int(h)


def _cover_filter(w, h):
    return f"scale={w}:{h}:force_original_aspect_ratio=increase:flags=lanczos,crop={w}:{h}"


def read_frames(path, start, count, w, h, gray=False):
    """count frames from `start` seconds at 24 fps, scaled to cover w x h and centre-cropped: a uint8 (C, T, H, W) tensor.
    A clip shorter than requested is padded with its last frame."""
    channels = 1 if gray else 3
    cmd = ["ffmpeg", "-v", "error", "-ss", f"{max(0.0, float(start)):.3f}", "-i", path, "-an", "-sn",
           "-vf", f"fps={FPS},{_cover_filter(w, h)}", "-frames:v", str(count),
           "-pix_fmt", "gray" if gray else "rgb24", "-f", "rawvideo", "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    frame_bytes = w * h * channels
    got = len(raw) // frame_bytes
    if got == 0:
        raise ValueError("No frames could be read at that start time.")
    frames = torch.frombuffer(bytearray(raw[:got * frame_bytes]), dtype=torch.uint8).view(got, h, w, channels)
    if got < count:
        frames = torch.cat((frames, frames[-1:].expand(count - got, -1, -1, -1)))
    return frames.permute(3, 0, 1, 2).contiguous(), got


def read_audio(path, start, seconds, out_wav):
    """The source soundtrack of the window as 32 kHz stereo float32 (samples, 2), also written to out_wav; None if silent."""
    cmd = ["ffmpeg", "-v", "error", "-y", "-ss", f"{max(0.0, float(start)):.3f}", "-t", f"{seconds:.3f}", "-i", path,
           "-vn", "-ac", "2", "-ar", str(AUDIO_RATE), "-c:a", "pcm_f32le", out_wav]
    if subprocess.run(cmd, capture_output=True).returncode != 0 or not os.path.exists(out_wav):
        return None
    import soundfile as sf
    audio, _ = sf.read(out_wav, dtype="float32", always_2d=True)
    return audio if audio.shape[0] else None


def fit_image(image, w, h):
    """A painted frame at the canvas size: scaled to cover and centre-cropped (frames painted at the source's
    proportions are simply resized)."""
    image = image.convert("RGB")
    scale = max(w / image.width, h / image.height)
    sw, sh = max(w, round(image.width * scale)), max(h, round(image.height * scale))
    image = image.resize((sw, sh), Image.Resampling.LANCZOS)
    left, top = (sw - w) // 2, (sh - h) // 2
    return image.crop((left, top, left + w, top + h))


def grab_frame(path, start, frame_no):
    """Source frame `frame_no` (1 = the first frame of the window) at full resolution, as a PIL image."""
    t = max(0.0, float(start)) + (max(1, int(frame_no)) - 1) / FPS
    cmd = ["ffmpeg", "-v", "error", "-ss", f"{t:.4f}", "-i", path, "-frames:v", "1", "-f", "image2pipe", "-vcodec", "png", "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    import io
    return Image.open(io.BytesIO(raw)).convert("RGB")


def latent_rows(w, h, frames, painted, pinned):
    """Packed sequence rows of one forward: text, target and source video, painted references, pinned keyframes, audio."""
    per_frame = (h // 32) * (w // 32)
    video = h3_pipeline.video_latent_frames(frames) * per_frame
    audio = round((frames + 1) / FPS * h3_pipeline.AUDIO_LATENT_FPS) * 2
    return 700 + 2 * video + (painted + pinned) * per_frame + audio


def step_seconds(rows, attention_speedup=1.0):
    """Seconds per step on Kaggle T4 x2 with exact attention: 24 x^2 + 7 x + 6, x = rows / 16k. The quadratic term is
    attention, divided by the speedup of the chosen attention."""
    x = rows / 16384
    return 24 * x * x / attention_speedup + 7 * x + 6


def needed_bytes(rows, long_video=False):
    """Working VRAM per GPU: activations per packed row (half the heads per GPU) + scratch. Estimates."""
    return rows * (48 if long_video else 110) * 1024 + 700 * 1024**2
# <<< section


# >>> section:prompt
def _clean(text):
    return " ".join(str(text or "").split()).strip().rstrip(".")


def _ts(seconds):
    seconds = max(0.0, seconds)
    return f"{int(seconds // 60):02d}:{seconds % 60:06.3f}"


def _picture_names(n):
    names = [f"<Picture {i + 1}>" for i in range(n)]
    return names[0] if n == 1 else ", ".join(names[:-1]) + " and " + names[-1]


def build_remake_prompt(mode, frames, n_pictures, pins, source, who, name, look, speech, change, kept, setting):
    """The six-section prompt Viggle Remake was trained on, written from a few plain fields. `pins`: one frame number
    (1 = first frame) per painted frame, or an empty list when the frames are not pinned. Swap: `who` is the person in
    the source, `name` / `look` the new character. Edit: `change` is the thing in the source, `kept` what it becomes
    (empty: removed), `setting` the rest of the scene."""
    n = max(1, int(n_pictures))
    pins = list(pins or [])[:n] if pins and len(pins) >= n else []
    duration = frames / FPS
    times = [_ts((p - 1) / FPS) for p in pins] if pins else []
    source = _clean(source) or "one continuous shot"
    pics = _picture_names(n)
    lines = []
    if mode == "swap":
        name = _clean(name) or "the new character"
        who = _clean(who) or "the person in the source"
        look = _clean(look) or "a new character whose whole look is shown in the painted frames"
        lines.append("subject_definitions:")
        lines.append(f"<Video 1> is the source video for the editing task: {source}.")
        if n == 1:
            lines.append(f"<Picture 1> is a frame of the target video, from [Shot 1]" + (f" at {times[0]}." if times else "."))
        else:
            where = (", " + ", ".join(f"<Picture {i + 1}> at {t}" for i, t in enumerate(times))) if times else ""
            lines.append(f"{pics} are {n} frames of the target video, from [Shot 1]{where}.")
        lines.append(f"<Subject 1> is {name} in {pics}: {look}. {name[0].upper() + name[1:]} replaces {who} in <Video 1>.")
        lines += ["", "summary:",
                  f"[video editing + character replacement] The target video is an edited version of <Video 1> in which {who} "
                  f"becomes <Subject 1>, {name}, looking exactly as shown in {pics}. All other people, the place and the "
                  f"soundtrack are unchanged; the shot, camera, motion and timing of <Video 1> are kept.",
                  "", "retention_analysis:",
                  "<Video 1> (source video editing): fully_preserved - the shot, the camera framing and movement, the place, "
                  "the people who stay, and the motion and timing of every performance are copied frame for frame."]
        for i in range(n):
            last = " A person shown in several pictures is the same person." if n > 1 and i == n - 1 else ""
            lines.append(f"<Picture {i + 1}> (appears in [Shot 1]): reference - a frame of the target video. The people in it, "
                         f"their faces and clothing, and the background, framing and lighting are taken from <Picture {i + 1}>.{last}")
        lines.append(f"<Subject 1> (appears in [Shot 1]): fully_preserved - {name}'s face, hair and whole outfit stay identical in every frame.")
        desc = [f"[Shot 1] <Subject 1>, {name}, appears exactly as in " + (pics if not times else
                ", ".join(f"<Picture {i + 1}> at {t}" for i, t in enumerate(times))) + ", with the motion, camera and timing of <Video 1>."]
        if speech:
            desc.append("<Subject 1> (S1) is speaking with natural lip movements synchronized, <d>[unclear]</d>.")
        desc.append(f"In every frame <Subject 1> is this same {name} exactly as in {pics}: the original look of {who} "
                    f"appears in no frame. Only <Subject 1> changes, and everything else stays as in <Video 1>.")
        lines += ["", "detailed_description:", "Realistic footage with natural light and colours, as in <Video 1>.", " ".join(desc)]
    else:
        kept_text = _clean(kept)
        change = _clean(change) or "the object to change"
        setting = _clean(setting) or "the place, the people and the light of <Video 1>"
        lines.append("subject_definitions:")
        lines.append("<Video 1> is the source video for the target video edit" + (f": {source}." if _clean(source) else "."))
        for i in range(n):
            lines.append(f"<Picture {i + 1}> is a frame of the target video, exactly as the target video shows it at that moment.")
        lines.append(f"<Subject 1> is {change} in <Video 1>.")
        lines.append(f"<Subject 2> is everything that stays as in <Video 1>: {setting}.")
        if kept_text:
            lines.append(f"<Subject 3> is the target video's version of it: {kept_text}.")
        result = f"it becomes <Subject 3>, {kept_text}" if kept_text else "it is gone in every frame"
        lines += ["", "summary:",
                  f"[video editing] The target video is an edited version of <Video 1>. {change[0].upper() + change[1:]}: {result}, "
                  "from the very first frame to the last, while <Subject 2> moves exactly as in <Video 1> with the same framing, "
                  "camera motion and timing.",
                  "", "retention_analysis:",
                  "<Video 1> (source video editing): fully_preserved - the framing and camera movement, every movement of the "
                  "people and their timing are copied frame for frame."]
        for i in range(n):
            if pins:
                lines.append(f"<Picture {i + 1}> (target keyframe): fully_preserved - the target video passes through this exact frame.")
            else:
                lines.append(f"<Picture {i + 1}> (reference): the look of the change is taken from <Picture {i + 1}>.")
        if kept_text:
            lines.append(f"<Subject 1> (appears in [Shot 1]): replaced - {change} is replaced by <Subject 3> in every frame; "
                         "nothing of it remains anywhere.")
        else:
            lines.append("<Subject 1> (appears in [Shot 1]): removed - it is completely removed in every frame, with no trace "
                         "or shadow of it; what was behind it is visible where it stood.")
        lines.append(f"<Subject 2> (appears in [Shot 1]): fully_preserved - {setting} stay exactly as in <Video 1>.")
        if kept_text:
            lines.append(f"<Subject 3> (appears in [Shot 1]): new - {kept_text}, exactly as the painted frames show it, in every frame.")
        desc = f"[Shot 1] From 00:00.000 to {_ts(duration)}, one continuous shot. "
        desc += (f"From the very first frame <Subject 3> is there, exactly as in {pics}, and it stays in every frame. "
                 if kept_text else f"{change[0].upper() + change[1:]} appears in no frame. ")
        desc += "<Subject 2> moves exactly as in <Video 1>."
        if times:
            desc += " " + " ".join(f"At {t} the scene matches <Picture {i + 1}>." for i, t in enumerate(times))
        lines += ["", "detailed_description:", "Clean, sharp footage with the light and colours of the target video.", desc]
    lines += ["", "overall_soundscape:", "The soundtrack of <Video 1>, unchanged.", "", "non_diegetic_music:", "N/A"]
    return "\n".join(lines)


PROMPT_SECTIONS = ("subject_definitions:", "summary:", "retention_analysis:", "detailed_description:", "overall_soundscape:", "non_diegetic_music:")


def check_prompt(prompt, n_pictures, frames):
    """Problems with a remake prompt that make the model ignore parts of it, as short messages."""
    problems = []
    missing = [s for s in PROMPT_SECTIONS if s not in prompt]
    if missing:
        problems.append("missing sections: " + ", ".join(missing))
    used = {int(m) for m in re.findall(r"<Picture (\d+)>", prompt)}
    if used and max(used) > n_pictures:
        problems.append(f"it mentions <Picture {max(used)}> but only {n_pictures} painted frame(s) are given")
    late = [t for t in re.findall(r"(\d\d):(\d\d\.\d{3})", prompt) if int(t[0]) * 60 + float(t[1]) > frames / FPS + 0.05]
    if late:
        problems.append(f"timestamps past the end of the {frames / FPS:.1f} s window: " + ", ".join(f"{a}:{b}" for a, b in late[:3]))
    if re.search(r"\bdim(ly)?\b", prompt, re.IGNORECASE):
        problems.append('the word "dim" darkens the whole render (write "low light")')
    return problems
# <<< section


# ==== Live progress (also polled by the page over plain HTTP, which survives tunnels dropping Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "video": None, "message": "", "final_n": 0}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def free_for_denoise():
    aux_offload.unload_all()
    gc.collect()
    frees = []
    for dev in DEVICES:
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
        frees.append(torch.cuda.mem_get_info(dev)[0])
    return frees


def save_mp4(video, path, audio_source=None, audio_start=0.0, fps=FPS):
    """Streams a uint8 [3, T, H, W] video to ffmpeg 16 frames at a time; the source's own soundtrack (original quality)
    is muxed in when there is one."""
    _, t, h, w = video.shape
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w}x{h}", "-r", str(fps), "-i", "-"]
    if audio_source:
        cmd += ["-ss", f"{audio_start:.3f}", "-t", f"{t / fps:.3f}", "-i", audio_source, "-map", "0:v:0", "-map", "1:a:0",
                "-c:a", "aac", "-b:a", "192k"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "17", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
            "-t", f"{t / fps:.3f}", path]  # the video's length (an AAC track ends a few ms early; -shortest would cut a frame)
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    try:
        for start in range(0, t, 16):
            frames = video[:, start:start + 16]
            if frames.dtype != torch.uint8:
                frames = frames.float().clamp(-1, 1).add(1).mul(127.5).round().to(torch.uint8)
            proc.stdin.write(frames.permute(1, 2, 3, 0).contiguous().numpy().tobytes())
        proc.stdin.close()
    except BrokenPipeError:
        pass
    if proc.wait() != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {proc.returncode}")


def save_side_by_side(source, start, frames, w, h, remake_path, out_path):
    """Source (left) and remake (right) in one MP4 with the remake's soundtrack, for before / after comparisons."""
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{start:.3f}", "-i", source, "-i", remake_path,
           "-filter_complex", f"[0:v]fps={FPS},{_cover_filter(w, h)},trim=end_frame={frames},setpts=PTS-STARTPTS[a];"
                              f"[1:v]setpts=PTS-STARTPTS[b];[a][b]hstack=inputs=2[v]",
           "-map", "[v]", "-map", "1:a?", "-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-pix_fmt", "yuv420p",
           "-c:a", "copy", "-t", f"{frames / FPS:.3f}", "-movflags", "+faststart", out_path]
    return subprocess.run(cmd, capture_output=True).returncode == 0


# ==== Encoded inputs are cached between videos ====
# Re-running on the same clip (another seed, a tweaked prompt or painted frame) reuses the VAE latents of the source
# video and of unchanged painted frames instead of encoding them again (the encode is deterministic: same input, same
# latents). The prompt encoding has its own cache in Wan2GP.
import collections
import hashlib

ENCODE_CACHE = collections.OrderedDict()


def attention_warning(int8):
    """A status line only when an INT8 run had to switch back to exact attention."""
    if not int8 or ATTENTION["fallback"] is None:
        return ""
    return "⚠️ Fast attention was not accurate enough on this video, so it was finished with exact attention"


def install_hooks(pipe):
    """The encode cache and the switch to per-GPU threads after the first transformer pass, on this pipeline."""
    orig_encode_video = pipe._encode_video
    orig_forward = pipe.transformer.forward

    def cached_encode_video(video, keep_all_latents=False):
        if video.is_cuda:
            return orig_encode_video(video, keep_all_latents=keep_all_latents)
        data = video.contiguous()
        digest = hashlib.blake2b(memoryview(data.view(torch.uint8).reshape(-1).numpy()), digest_size=16).hexdigest()
        vae = pipe.vae
        key = (id(pipe), digest, str(video.dtype), tuple(video.shape), bool(keep_all_latents), getattr(vae, "use_tiling", None),
               getattr(vae, "tile_sample_min_height", None), getattr(vae, "tile_sample_min_width", None))
        if key in ENCODE_CACHE:
            ENCODE_CACHE.move_to_end(key)
            return ENCODE_CACHE[key].clone()
        latent = orig_encode_video(video, keep_all_latents=keep_all_latents)
        ENCODE_CACHE[key] = latent.clone()
        while len(ENCODE_CACHE) > 8:
            ENCODE_CACHE.popitem(last=False)
        return latent

    def forward(*args, **kwargs):
        out = orig_forward(*args, **kwargs)
        TP_THREADS["warm"] = True  # the first pass of the session (one thread) has compiled every kernel on both GPUs
        return out

    pipe._encode_video = cached_encode_video
    pipe.transformer.forward = forward


if "pipeline" in globals():
    install_hooks(pipeline)


@torch.inference_mode()
def run_remake(kwargs, steps):
    pipeline._interrupt = False
    t0 = time.time()
    timings = {"encode": 0.0, "denoise": 0.0, "decode": 0.0}
    phase = {"name": "prep", "t": time.time(), "step_t": time.time()}

    def callback(step, latent=None, is_start=False, override_num_inference_steps=None, progress_unit=None,
                 progress_title=None, **kw):
        now = time.time()
        if progress_unit == "layers":  # Qwen3-VL prompt encoding
            total = override_num_inference_steps or 50
            BAR.start("text", "Prompt encoding", total, "layers")
            BAR.update(max(0, step + 1), total)
            report(0.10 + 0.05 * min(1.0, (step + 1) / total), f"📝 Encoding the prompt {max(0, step + 1)}/{total}")
            return
        if progress_unit == "tiles":  # VAE encoding of the source video and painted frames
            total = override_num_inference_steps or 1
            done = min(total, max(0, step + 1) * PARALLEL_ENC["scale"])  # both GPUs encode; GPU 1's tiles are counted
            BAR.start("encode", "Encoding source video", total, "tiles")
            BAR.update(done, total)
            report(0.02 + 0.08 * done / total, f"🎞️ Encoding the source video {done}/{total}")
            return
        if is_start:
            timings["encode"] = now - t0
            phase.update(name="denoise", t=now, step_t=now, total=override_num_inference_steps or steps)
            BAR.start("denoise", "Denoising", phase["total"], "steps")
            report(0.15, f"🎬 Denoising step 0/{phase['total']}")
            return
        total = phase.get("total", steps)
        dt = now - phase["step_t"]
        phase["step_t"] = now
        done = step + 1
        BAR.update(done, total)
        report(0.15 + 0.73 * done / total, f"🎬 Denoising step {done}/{total} ({dt:.0f}s/step)")

    def set_status(text):
        if "Decod" in str(text) and phase["name"] == "denoise":
            timings["denoise"] = time.time() - phase["t"]
            phase.update(name="decode", t=time.time())

    report(0.01, "⚡ Preparing...")
    BAR.start("prep", "Preparing inputs")
    try:
        result = pipeline.generate(callback=callback, set_progress_status=set_status, **kwargs)
    finally:
        BAR.finish()
    if result is None:
        return None, timings
    if phase["name"] == "decode":
        timings["decode"] = time.time() - phase["t"]
    elif phase["name"] == "denoise":
        timings["denoise"] = phase["step_t"] - phase["t"]
        timings["decode"] = time.time() - phase["step_t"]
    timings["total"] = time.time() - t0
    return result["x"], timings


def on_main_gpu(fn):
    """CUDA's current device is per thread: Gradio workers start on GPU 0. Run handlers with GPU 1 current, the device
    the sampler, text encoder and VAEs use for "cuda"."""
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        with torch.cuda.device(MAIN_DEV):
            return fn(*args, **kwargs)
    return wrapped


def _path(value):
    if value is None:
        return None
    if isinstance(value, dict):
        value = value.get("video") or value.get("path") or value.get("name")
        if isinstance(value, dict):
            value = value.get("path")
    return value if isinstance(value, str) and value else getattr(value, "name", None)


@on_main_gpu
def generate_remake(source, start_s, duration, resolution, pin_on, img1, img2, img3, img4, pin1, pin2, pin3, pin4,
                    prompt, mask, seed, side_by_side, attention=ATTENTION_EXACT):
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    try:
        source = _path(source)
        if not source or not os.path.exists(source):
            return None, None, "❌ Upload a source video first."
        painted = [(img, pin) for img, pin in zip((img1, img2, img3, img4), (pin1, pin2, pin3, pin4)) if img is not None]
        if not painted:
            return None, None, ("❌ Add at least one painted frame: a frame of the source video edited in an image editor to show "
                                "the new look (use 🎞️ Grab a frame to get one).")
        prompt = (prompt or "").strip()
        if not prompt:
            return None, None, "❌ Write the prompt (or fill the ✍️ Prompt builder and press Build)."
        info = probe_video(source)
        start = max(0.0, float(start_s or 0))
        frames = DURATIONS.get(duration, 124)
        available = int((info["duration"] - start) * FPS + 1e-6) if info["duration"] else frames
        if available < frames:
            fit = [n for n in WINDOW_FRAMES if n <= available]
            if not fit:
                return None, None, (f"❌ Only {available / FPS:.1f} s of video after {start:.1f} s; Viggle Remake needs at least "
                                    f"{VIGGLE_REMAKE_MIN_FRAMES / FPS:.2f} s ({VIGGLE_REMAKE_MIN_FRAMES} frames). Pick an earlier start.")
            frames = fit[-1]
        w, h = canvas_size(RESOLUTIONS.get(resolution, 480), info["width"], info["height"])
        pins = []
        if pin_on:
            for index, (_, pin) in enumerate(painted):
                if not pin or int(pin) < 1 or int(pin) > frames:
                    return None, None, (f"❌ Painted frame {index + 1}: its pin must be the frame number it was painted from, "
                                        f"1 to {frames} (frame 1 = the first frame of the window).")
                pins.append(int(pin))
            if pins != sorted(pins) or len(set(pins)) != len(pins):
                return None, None, "❌ Pins must increase from painted frame 1 to 4: put the painted frames in time order."
        problems = check_prompt(prompt, len(painted), frames)
        if seed is None or int(seed) < 0:
            seed = random.randint(0, 2**31 - 1)
        seed = int(seed)

        report(0.01, "🎞️ Reading the source video...")
        video, got = read_frames(source, start, frames, w, h)
        tmp = os.path.join(OUTPUT_DIR, ".gradio_cache")
        os.makedirs(tmp, exist_ok=True)
        audio = read_audio(source, start, frames / FPS, os.path.join(tmp, "source_window.wav")) if info["audio"] else None
        mask_frames = None
        mask = _path(mask)
        if mask:
            mask_frames, _ = read_frames(mask, start, frames, w, h, gray=True)
        images = [fit_image(img, w, h) for img, _ in painted]

        rows = latent_rows(w, h, frames, len(images), len(pins))
        frees = free_for_denoise()
        LONG_VIDEO["on"] = needed_bytes(rows) > min(frees)
        if needed_bytes(rows, LONG_VIDEO["on"]) > min(frees):
            return None, None, (f"❌ {w}x{h} for {frames / FPS:.1f} s needs about {_gb(needed_bytes(rows, True))} of working VRAM per "
                                f"GPU even in long-video mode, but only {_gb(frees[0])} / {_gb(frees[1])} is free. Pick a shorter "
                                f"duration or a lower resolution.")
        int8 = attention == ATTENTION_INT8 and ATTENTION["int8"] is not None
        estimate = VIGGLE_REMAKE_STEPS * step_seconds(rows, attention_speedup(attention)) * (1.25 if LONG_VIDEO["on"] else 1.0)

        kwargs = dict(input_prompt=prompt, input_frames=video, frame_num=frames, height=h, width=w, fps=FPS,
                      shift=VIGGLE_REMAKE_SHIFT, sampling_steps=VIGGLE_REMAKE_STEPS, seed=seed, sample_solver="euler",
                      guide_phases=1, image_refs_relative_size=100,
                      video_prompt_type=("F" if pins else "") + "IV" + ("A" if mask_frames is not None else "U"),
                      audio_prompt_type="K" if audio is not None else "")
        if pins:
            kwargs.update(frames_to_inject=images, frames_relative_positions_list=[p - 1 for p in pins], input_ref_images=[])
        else:
            kwargs["input_ref_images"] = images
        if mask_frames is not None:
            kwargs["input_masks"] = mask_frames
        if audio is not None:
            kwargs.update(input_waveform=audio, input_waveform_sample_rate=AUDIO_RATE)

        notes = ((" | mask" if mask_frames is not None else "") + (" | silent source" if audio is None else "")
                 + (" | fast attention" if int8 else ""))
        print(f"🎬 {MODE.capitalize()} | {w}x{h} | {frames / FPS:.1f}s from {start:.1f}s | {len(images)} painted frame(s)"
              f"{' pinned' if pins else ''}{notes} | seed {seed} | est. ~{_fmt_secs(estimate)} denoising")
        sys.stdout.flush()

        for dev in DEVICES:
            torch.cuda.reset_peak_memory_stats(dev)
        PEAKS.clear()
        # INT8 attention is checked against exact attention on this video in the first step (blocks 1 and 26, GPU 0)
        ATTENTION.update(mode="int8" if int8 else "sdpa", checks={0, NUM_BLOCKS // 2} if int8 else set(),
                         fallback=None)
        result, timings = run_remake(kwargs, VIGGLE_REMAKE_STEPS)
        if result is None:
            return None, None, "⏹️ Generation stopped."
        report(0.97, "💾 Saving MP4...")
        stamp = time.strftime("%Y%m%d_%H%M%S")
        out_path = os.path.join(OUTPUT_DIR, f"viggle_{MODE}_{stamp}_seed{seed}.mp4")
        BAR.start("save", "Saving MP4")
        save_mp4(result, out_path, source if audio is not None else None, start)
        compare_path = None
        if side_by_side:
            compare_path = os.path.join(OUTPUT_DIR, "compare", f"viggle_{MODE}_{stamp}_seed{seed}_side_by_side.mp4")
            os.makedirs(os.path.dirname(compare_path), exist_ok=True)
            if not save_side_by_side(source, start, frames, w, h, out_path, compare_path):
                compare_path = None
        BAR.finish()
        total = timings.get("total", 0.0)
        per_step = timings["denoise"] / VIGGLE_REMAKE_STEPS
        status = "\n".join([
            f"✅ Done in {_fmt_secs(total)}  (Viggle Remake {MODE.capitalize()})",
            f"⚡ Inputs + prompt {timings['encode']:.0f}s | Denoising {timings['denoise']:.0f}s ({per_step:.1f}s/step) | VAE {timings['decode']:.0f}s",
            f"📐 {w}x{h} | {frames} frames @ 24 fps ({frames / FPS:.1f}s from {start:.1f}s) | "
            + ("source soundtrack kept" if audio is not None else "silent source"),
            f"🖼️ {len(images)} painted frame(s)" + (f", pinned at {', '.join(map(str, pins))}" if pins else ", look only (not pinned)")
            + (" | 🎭 mask" if mask_frames is not None else ""),
            f"🎲 Seed {seed} | 6 Euler steps, Viggle's {MODE} schedule | "
            + ("⚡ Fast attention" if int8 and ATTENTION["fallback"] is None else "exact attention"),
            f"🧠 Peak VRAM per GPU: "
            + (f"denoise {_gb(PEAKS['denoise'][0])} / {_gb(PEAKS['denoise'][1])}, " if "denoise" in PEAKS else "")
            + f"decode {_gb(torch.cuda.max_memory_allocated(0))} / {_gb(torch.cuda.max_memory_allocated(1))}",
            f"📁 {out_path}",
        ] + [line for line in (attention_warning(int8),) if line] + [f"⚠️ Prompt: {p}" for p in problems])
        print(f"✅ Done in {_fmt_secs(total)} ({per_step:.0f}s/step) → {os.path.basename(out_path)}")
        drop_file_cache(OUTPUT_DIR)
        sys.stdout.flush()
        return out_path, compare_path, status
    except torch.OutOfMemoryError:
        traceback.print_exc()
        return None, None, "❌ Out of GPU memory. Pick a lower resolution or a shorter duration and try again."
    except Exception as e:
        traceback.print_exc()
        if "CUDA error" in str(e):  # e.g. an illegal memory access: the GPU context is unusable from here on
            print("❌ The GPUs are in an error state. Re-run Step 2 to restart the engine (the models reload in ~4 min).")
            return None, None, f"❌ GPU error: {e}\nRe-run Step 2 to restart the engine."
        return None, None, f"❌ Error: {e}"
    finally:
        ATTENTION.update(mode="sdpa", checks=set())
        BAR.abort()
        PROGRESS["active"] = False
        for key in _tp_cache:  # last hidden state / rope / temb copies held on both GPUs
            _tp_cache[key] = None
        gc.collect()
        for dev in DEVICES:
            with torch.cuda.device(dev):
                torch.cuda.empty_cache()


# ==== Generation as a background job ====
# A clip takes 10-40 minutes, longer than the Gradio share / Cloudflare tunnels reliably keep one event stream open.
# The Generate button only starts a job and returns; the page script polls /aiq_progress over plain HTTP and clicks a
# hidden button to fetch the result when a new one is ready, so a reloaded page or a dropped connection loses nothing.
JOB = {"id": 0, "done": 0, "running": False, "video": None, "compare": None, "status": "Ready."}


def _job_worker(job_id, args):
    try:
        video, compare, status = generate_remake(*args)
    except BaseException as e:  # generate_remake reports its own errors; this is a last resort
        video, compare, status = None, None, f"❌ Error: {e}"
    JOB.update(running=False, done=job_id, video=video, compare=compare, status=status)
    PROGRESS.update(active=False, finished=time.time(), video=os.path.basename(video) if video else None,
                    message=status.splitlines()[0] if status else "", final_n=job_id)


def start_generation(*args):
    if JOB["running"]:
        return "⏳ A video is already being generated. It appears here as soon as it is ready."
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None, message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return ("🚀 Started. Progress shows at the top of the page and the video appears below when it is ready. "
            "Reloading the page or a dropped connection does not stop the job.")


def deliver_result():
    if JOB["running"]:
        return gr.skip(), gr.skip(), f"⏳ {PROGRESS['text'] or 'Working...'}"
    if not JOB["done"]:
        return gr.skip(), gr.skip(), JOB["status"]
    return JOB["video"], JOB["compare"], JOB["status"]


def stop_generation():
    if not JOB["running"]:
        return "Nothing is running."
    pipeline._interrupt = True
    return "⏹️ Stopping after the current step..."


def load_latest_video():
    videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
    if not videos:
        return None, None, f"No videos in {OUTPUT_DIR} yet."
    compare = os.path.join(OUTPUT_DIR, "compare", os.path.basename(videos[-1])[:-4] + "_side_by_side.mp4")
    return videos[-1], compare if os.path.exists(compare) else None, f"📁 Loaded the latest video: {videos[-1]}"


def grab_frame_ui(source, start_s, frame_no, slot, _pin_on):
    source = _path(source)
    if not source:
        return gr.skip(), "❌ Upload a source video first.", *([gr.skip()] * MAX_PAINTED)
    try:
        image = grab_frame(source, start_s or 0, frame_no or 1)
    except Exception as e:
        return gr.skip(), f"❌ Could not read that frame: {e}", *([gr.skip()] * MAX_PAINTED)
    slot_index = int(str(slot).split()[-1]) - 1
    frame_no = max(1, int(frame_no or 1))
    pins = [gr.skip()] * MAX_PAINTED
    pins[slot_index] = frame_no
    return (image, f"✅ Frame {int(frame_no or 1)} ({image.width}x{image.height}). Download it, paint the change in an image editor at "
                   f"the same size, then drop the result into Painted frame {slot_index + 1} (its pin is set to {int(frame_no or 1)}).",
            *pins)


def source_info(source, start_s, duration, resolution, attention=ATTENTION_EXACT):
    source = _path(source)
    if not source:
        return "ℹ️ Upload a source video (24 fps or any rate; it is converted to 24 fps)."
    try:
        info = probe_video(source)
    except Exception as e:
        return f"❌ Unreadable video: {e}"
    frames = DURATIONS.get(duration, 124)
    w, h = canvas_size(RESOLUTIONS.get(resolution, 480), info["width"], info["height"])
    rows = latent_rows(w, h, frames, 1, 1)
    est = VIGGLE_REMAKE_STEPS * step_seconds(rows, attention_speedup(attention)) * (1.25 if needed_bytes(rows) > 5.0 * 1024**3 else 1.0)
    left = info["duration"] - float(start_s or 0)
    warn = f" ⚠️ only {left:.1f} s left after the start: the window will be shortened." if left * FPS < frames else ""
    return (f"ℹ️ Source {info['width']}x{info['height']}, {info['duration']:.1f} s, {info['fps']:.0f} fps, "
            f"{'with sound' if info['audio'] else 'silent'} → render {w}x{h}, {frames} frames. "
            f"Estimated denoising ≈ {_fmt_secs(est)} + ~1-2 min encode/decode.{warn}")


def build_prompt_ui(duration, pin_on, img1, img2, img3, img4, pin1, pin2, pin3, pin4, source_desc, who, name, look, speech,
                    change, result, setting):
    slots = [(img, pin) for img, pin in zip((img1, img2, img3, img4), (pin1, pin2, pin3, pin4)) if img is not None]
    pins = [int(pin) for _, pin in slots] if pin_on and slots and all(pin for _, pin in slots) else []
    return build_remake_prompt(MODE, DURATIONS.get(duration, 124), max(1, len(slots)), pins, source_desc, who, name, look,
                               speech, change, result, setting)


# Viggle's official examples with complete inputs (source clip with sound, two painted frames, the prompt): one per mode.
# Each mode gets the official example as published (9.4 s) and a quick test on the same inputs: the first 5.2 s, the
# first painted frame only, and the official prompt trimmed by AIQUEST to that window. Viggle's pins count from 0
# (frame 12 = 0.5 s); the pin fields here count from 1, so they are shown as +1.
VIGGLE_REVISION = "7e5be143702dd53b9154482de673b8519ed2d4d6"  # Viggle/Viggle-Remake, 7 Nov 2026
QUICK_PROMPTS = {"swap": """subject_definitions:
<Video 1> is the source video for the editing task: one continuous 5.2 s handheld take on a sunny park road in London in which a pizza maker smiles at the camera, tosses a big sheet of white pizza dough high into the air, catches it and stretches it in front of the camera.
<Picture 1> is a frame of the target video, from [Shot 1].
<Subject 1> is Pete in <Picture 1>: a person wholly hidden inside a full-body plush pineapple mascot costume, with a big round golden-yellow pineapple head with a raised diamond-pattern rind and a spiky crown of stiff green pineapple leaves on top, two big round black-and-white cartoon eyes and a wide stitched black smile on the front of the pineapple where the face would be, a padded golden-yellow bodysuit with the same diamond-pattern pineapple rind over the torso, the arms and the legs, bright-green padded five-finger gloves and bright-green padded boots. The costume covers the whole body; no human face and no human hair show. He replaces the pizza maker in the grey camouflage cap, grey hoodie, neck gaiter and jeans in <Video 1>.

summary:
[video editing + character replacement] The target video is an edited version of <Video 1> in which the pizza maker becomes <Subject 1>, Pete the pineapple mascot, looking exactly as he does in <Picture 1>, tossing and stretching the same sheet of white pizza dough. The few tiny distant people in the park, the road, the lawns, the trees, the benches, the tent and the red fire-point trolley are unchanged. The single continuous shot, the camera, the motion, the timing and the actions of <Video 1> are kept.

retention_analysis:
<Video 1> (source video editing): fully_preserved - the single continuous shot with no cut, the handheld camera framing and movement including the tilt up to the park, the location, the sunny daylight, the background people, and the motion and timing of every toss, catch and stretch of the dough are copied frame for frame.
<Picture 1> (appears in [Shot 1]): reference - a frame of the target video. Pete's whole figure from slightly above, his pineapple head, bodysuit, green gloves and green boots, the dough in his gloves, and the asphalt, kerb and grass are taken from <Picture 1>.
<Subject 1> (appears in [Shot 1]): fully_preserved - the round golden-yellow diamond-pattern pineapple head with the green leaf crown, the cartoon eyes and the stitched smile, the matching padded bodysuit over the torso, arms and legs, the bright-green five-finger gloves and the bright-green boots stay identical in every frame, also during the toss and when he is close to the lens.

detailed_description:
The target video is a realistic sunny outdoor handheld scene in a London park with bright daylight and natural colours.
[Shot 1] At 00:00.000 the camera looks down from slightly above at <Subject 1>, Pete the pineapple mascot, standing on the asphalt road with his whole figure in view, his pineapple face turned up to the camera, a big sheet of white pizza dough hanging from both green gloves in front of his waist, matching <Picture 1>. He swings the dough from side to side, and at about 00:01.400 he raises both arms and tosses it high into the air; the camera tilts up with it, and at about 00:01.900 the dough sheet spins high above the sunny park while he looks up with one arm raised. At about 00:02.600 he catches it in both gloves, now in a medium view with the lawns, trees, benches and the red fire-point trolley behind him and a few tiny distant people on the lawns. From 00:03.000 he stretches and swings the dough sheet in front of him, and at about 00:04.600 he steps close to the camera holding the big sheet up to the left, glancing at the lens. The camera is handheld throughout.
In every frame, from the first to the last, the pizza maker is this same pineapple mascot exactly as in <Picture 1>: no human face, no human hair, no grey cap, no neck gaiter, no grey hoodie, no jeans, no sneakers, no printed text and no logo at any moment. Only one sheet of dough exists. Only this one figure changes, and everything else stays as in <Video 1>.

overall_soundscape:
The source soundtrack continues throughout, with his voice as he talks to the camera, open park ambience with light wind, and the soft flap of the dough as it is tossed and stretched.

non_diegetic_music:
N/A""", "edit": """subject_definitions:
<Video 1> is the source video for the target video edit.
<Picture 1> is a frame of the target video, exactly as the target video shows it at that moment.
<Subject 1> is the wind turbines in <Video 1>: the tall white three-bladed wind turbine standing just left of the farm track close to the camera, with its white tower, nacelle and slowly turning blades, plus the two smaller white turbines further out on the pasture at the left and at the right, and the long dark shadows the turbines cast on the grass.
<Subject 2> is the setting from <Video 1>: a sunny winter morning over open farmland - a small dark stone farm building with a slate roof and a fenced yard at the lower left, a pale dirt farm track running away from the camera to a crossroads of farm tracks, wide flat green pastures with long tree shadows in the foreground, a hazy horizon and a blue sky with soft white clouds.

summary:
[video editing] The target video is an edited version of <Video 1>. The drone rises slowly and drifts forward over the farm building and the farm track exactly as in <Video 1>, but there are no wind turbines at all: the track and the crossroads run into open, empty green pasture under a blue sky with clouds, with exactly the same framing, drone motion, light and timing as <Video 1>.

retention_analysis:
<Video 1> (source video editing): fully_preserved - the framing, the rising and forward-drifting drone move, the farm building, the farm track, the crossroads, the pastures, the tree shadows, the horizon and the sky are copied frame for frame.
<Picture 1> (target keyframe): fully_preserved - the target video passes through this exact frame.
<Subject 1> (appears in [Shot 1]): removed - every wind turbine is completely removed in every frame: no white tower, no nacelle and no blades left of the track, no small turbine on the left or the right pasture, and no turbine shadow on the grass; blue sky and soft clouds are visible where the towers and blades stood, and plain green grass where their bases and shadows were.
<Subject 2> (appears in [Shot 1]): fully_preserved - the farm building, the fenced yard, the farm track, the crossroads, the pastures, the tree shadows, the hazy horizon and the sky are retained.

detailed_description:
The target video is a clean, sharp aerial shot in calm countryside travel style, low winter-morning sunlight, long soft shadows, a smooth slow drone move. No text and no lettering anywhere.
[Shot 1] From 00:00.000 to 00:05.167, the drone rises slowly and drifts forward over <Subject 2> exactly as in <Video 1>: at first it is low beside the dark stone farm building at the lower left, with the pale farm track running away from the camera across the green pasture and long tree shadows in the foreground. There is no wind turbine anywhere in the frame: above the track and the pasture only blue sky and soft white clouds. At 00:00.500 the scene matches <Picture 1>. The drone keeps rising, the farm building sliding down towards the lower left corner, and the crossroads of farm tracks and the wide flat pastures come into view, still with no turbine anywhere in the frame.

overall_soundscape:
The soundtrack of <Video 1>, unchanged.

non_diegetic_music:
N/A"""}
OFFICIAL_EXAMPLES = {
    "swap": {"name": "swap-pineapple-mascot", "pins": [13, 185], "what": "🍍 A pizza maker becomes a pineapple mascot"},
    "edit": {"name": "edit-remove-turbines", "pins": [13, 191], "what": "🌬️ The wind turbines vanish from a drone shot"},
}[MODE]


def ensure_examples():
    """The official example files of this mode (Step 1 downloads them; fetched here if they are missing)."""
    name = OFFICIAL_EXAMPLES["name"]
    files = [f"examples/inputs/{name}/{n}" for n in ("source.mp4", "ref1.png", "ref2.png")] + [f"examples/prompts/{name}.txt"]
    missing = [f for f in files if not os.path.exists(os.path.join(os.path.dirname(EXAMPLES_DIR), f))]
    if not missing:
        return True
    try:
        from huggingface_hub import hf_hub_download
        for f in missing:
            hf_hub_download("Viggle/Viggle-Remake", f, revision=VIGGLE_REVISION, local_dir=os.path.dirname(EXAMPLES_DIR))
        return True
    except Exception as e:
        print(f"⚠️ The official examples could not be downloaded ({str(e).splitlines()[0][:120]}); the UI shows none")
        return False


def official_example_rows():
    """Rows for gr.Examples: [source, start, duration, pin on, painted 1, painted 2, pin 1, pin 2, prompt, resolution]."""
    ex = OFFICIAL_EXAMPLES
    folder = os.path.join(EXAMPLES_DIR, "inputs", ex["name"])
    source, ref1, ref2 = (os.path.join(folder, n) for n in ("source.mp4", "ref1.png", "ref2.png"))
    prompt_file = os.path.join(EXAMPLES_DIR, "prompts", ex["name"] + ".txt")
    if not all(os.path.exists(f) for f in (source, ref1, ref2, prompt_file)):
        return [], []
    with open(prompt_file, encoding="utf-8") as f:
        prompt = f.read().strip()
    full = next(k for k, v in DURATIONS.items() if v == 226)
    quick = next(k for k, v in DURATIONS.items() if v == 124)
    rows = [[source, 0, full, True, ref1, ref2, ex["pins"][0], ex["pins"][1], prompt, DEFAULT_RESOLUTION],
            [source, 0, quick, True, ref1, None, ex["pins"][0], None, QUICK_PROMPTS[MODE], DEFAULT_RESOLUTION]]
    def minutes(frames, painted):  # total time at 480p with the default attention: denoising + ~1.5 min for the rest
        rows = latent_rows(832, 480, frames, painted, painted)
        return round(VIGGLE_REMAKE_STEPS * step_seconds(rows, attention_speedup(ATTENTION_DEFAULT)) / 60 + 1.5)
    labels = [f"{ex['what']}: official example, 9.4 s, 2 painted frames (~{minutes(226, 2)} min)",
              f"⚡ {ex['what'][2:].strip()}: quick test, first 5.2 s, 1 painted frame (~{minutes(124, 1)} min)"]
    return rows, labels


ensure_examples()


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#aiq-deliver { display: none !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.mode-badge { text-align: center; font-weight: 700; font-size: 15px; margin: 0 0 10px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎬 Viggle Remake: Video-to-Video</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
STATUS_HTML = (f'<div class="status-line">⚡ {INT8_LABEL} &nbsp;|&nbsp; {ATTENTION_LABEL} &nbsp;|&nbsp; MiniMax H3 20B tensor-parallel on 2 x T4 '
               f'&nbsp;|&nbsp; 6 steps &nbsp;|&nbsp; Powered by Viggle Remake</div>')
MODE_HTML = f'<div class="mode-badge">Mode: {MODE_LABEL} &nbsp;·&nbsp; <span style="font-weight:400">switch with REMAKE_MODE in Step 1</span></div>'
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved<br>'
               'Powered by <a href="https://huggingface.co/Viggle/Viggle-Remake" target="_blank">Viggle Remake</a> on MiniMax H3 · '
               'only change footage you have the rights to, never put a real person into a video without their agreement, and label results as AI-generated.</div>')

PROGRESS_HEAD = """
<script>
(function () {
  // Progress overlay fed by plain HTTP polling: it keeps working when a tunnel drops Gradio's event stream. When a new
  // result is ready it clicks the hidden deliver button once, which fetches the videos (no timer re-rendering the page).
  var box = null, fill = null, label = null, hideAt = 0, lastFinal = null;
  function build() {
    box = document.createElement("div");
    box.style.cssText = "position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:99999;width:min(560px,92vw);" +
      "background:#1f2937;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.25);padding:10px 14px;display:none;" +
      "font-family:Inter,sans-serif;color:#fff;font-size:13px";
    label = document.createElement("div");
    label.style.cssText = "margin-bottom:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis";
    var track = document.createElement("div");
    track.style.cssText = "height:8px;background:rgba(255,255,255,.18);border-radius:6px;overflow:hidden";
    fill = document.createElement("div");
    fill.style.cssText = "height:100%;width:0;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);transition:width .6s ease";
    track.appendChild(fill); box.appendChild(label); box.appendChild(track); document.body.appendChild(box);
  }
  async function tick() {
    try {
      var r = await fetch("aiq_progress?t=" + Date.now(), {cache: "no-store"});
      var p = await r.json();
      if (!box && document.body) build();
      if (lastFinal === null) { lastFinal = p.final_n; }
      var deliver = document.getElementById("aiq-deliver");
      if (deliver && p.final_n && p.final_n !== lastFinal && !p.active) { lastFinal = p.final_n; deliver.click(); }
      if (box) {
        if (p.active) {
          box.style.display = "block"; hideAt = 0;
          fill.style.width = Math.round(p.fraction * 100) + "%";
          label.textContent = "⏳ " + p.text + " · " + Math.round(p.elapsed) + "s";
        } else if (box.style.display === "block") {
          if (!hideAt) {
            fill.style.width = "100%";
            label.textContent = (p.video ? "✅ Done in " : "⚠️ Finished after ") + Math.round(p.elapsed) + "s ";
            if (p.video) {
              var link = document.createElement("a");
              link.href = "aiq_video?t=" + Date.now(); link.target = "_blank";
              link.style.cssText = "color:#c7d2fe;font-weight:600";
              link.textContent = "· ▶ open / download the video";
              label.appendChild(link);
            } else if (p.message) { label.textContent += "· " + p.message; }
            hideAt = Date.now() + 60000;
          } else if (Date.now() > hideAt) { box.style.display = "none"; }
        }
      }
    } catch (e) {}
    setTimeout(tick, 1000);
  }
  tick();
})();
</script>
"""

HOW_TO = {
    "swap": ("**How it works:** upload a clip → grab a frame where the person is clearly visible → repaint that frame in any image "
             "editor (keep the pose, framing and size; change face, hair, outfit) → drop it into *Painted frame 1* → build or write "
             "the prompt → Generate. The motion, camera, timing and **original soundtrack** are kept. Every replaced person must "
             "appear in a painted frame; pin every painted frame in Swap. Faces should be large and facing the camera."),
    "edit": ("**How it works:** upload a clip → grab a frame → repaint the change in an image editor (remove or add an object, "
             "new weather, light or setting; keep the framing and size) → drop it into *Painted frame 1* → build or write the "
             "prompt → Generate. People, motion, camera, timing and the **original soundtrack** are kept. Pin the painted frames "
             "when the clip has cuts or the exact composition matters; add a second painted frame for shots over ~8 s."),
}[MODE]

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=PROGRESS_HEAD, title="Viggle Remake | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.HTML(MODE_HTML)
    gr.Markdown(HOW_TO)

    with gr.Row():
        with gr.Column(scale=3):
            source = gr.Video(label="🎞️ Source video (with its soundtrack)", sources=["upload"], height=280)
            source_note = gr.Markdown("ℹ️ Upload a source video (24 fps or any rate; it is converted to 24 fps).")
        with gr.Column(scale=2):
            start_s = gr.Number(value=0, label="⏩ Start at (seconds)", info="The window starts here in the source")
            duration = gr.Dropdown(list(DURATIONS), value=DEFAULT_DURATION, label="⏱️ Window length",
                                   info="3.8 s to 14.4 s per render. Longer windows take much longer")
            resolution = gr.Dropdown(list(RESOLUTIONS), value=DEFAULT_RESOLUTION, label="📐 Resolution",
                                     info="Keeps the source's proportions. 480p is ~4x faster than 720p")
            seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")
            attention = gr.Radio(ATTENTION_CHOICES, value=ATTENTION_DEFAULT, label="⚡ Attention",
                                 info=("Fast: Comfy Kitchen INT8 attention, about twice as fast. Exact: full precision"
                                       if ATTENTION["int8"] is not None else "Exact: full precision (Fast is not available on this GPU)"))

    with gr.Accordion("🎞️ Grab a frame to paint", open=False):
        gr.Markdown("Pick the frame where the subject is large and clear. Download it, repaint it in any image editor at the same "
                    "size (ChatGPT / Gemini image edit, Qwen Image Edit, Photoshop...), then drop the result into a painted-frame slot.")
        with gr.Row():
            grab_no = gr.Number(value=1, precision=0, label="Frame number (1 = first frame of the window)")
            grab_slot = gr.Dropdown([f"Painted frame {i + 1}" for i in range(MAX_PAINTED)], value="Painted frame 1",
                                    label="Set the pin of")
            grab_btn = gr.Button("🎞️ Grab frame")
        grabbed = gr.Image(label="Source frame (download, paint, upload below)", type="pil", height=260, interactive=False)
        grab_note = gr.Markdown("")

    gr.Markdown("### 🖼️ Painted frames (`<Picture 1>` … `<Picture 4>` in the prompt)")
    pin_on = gr.Checkbox(value=MODE == "swap", label="📌 Pin painted frames at their frame numbers",
                         info="The video passes through each painted frame at that moment (recommended for Swap and for clips with cuts)")
    with gr.Row():
        imgs, pins = [], []
        for _i in range(MAX_PAINTED):
            with gr.Column(min_width=160):
                imgs.append(gr.Image(type="pil", label=f"Painted frame {_i + 1}", height=170))
                pins.append(gr.Number(value=None, precision=0, label=f"Pin: frame # of painted frame {_i + 1}"))  # no minimum: empty boxes send 0

    prompt = gr.Textbox(label="📝 Prompt (six sections, describe the finished video)", lines=12,
                        placeholder="subject_definitions:\n<Video 1> is the source video ...\n\nsummary:\n...\n\nretention_analysis:\n...\n\n"
                                    "detailed_description:\n...\n\noverall_soundscape:\nThe soundtrack of <Video 1>, unchanged.\n\nnon_diegetic_music:\nN/A")
    with gr.Accordion("✍️ Prompt builder: fill a few fields, get a ready six-section prompt", open=True):
        source_desc = gr.Textbox(label="What the source video shows", lines=1,
                                 placeholder="one 5 s handheld take of a woman in a green coat crossing a station hall")
        with gr.Group(visible=MODE == "swap"):
            with gr.Row():
                who = gr.Textbox(label="Who is replaced (their look in the source)", placeholder="the woman in the green coat with long dark hair")
                name = gr.Textbox(label="New character's name", placeholder="Mara")
            look = gr.Textbox(label="New character's whole look (face, hair, skin, outfit down to the shoes)",
                              placeholder="a tall woman with short silver hair, in a rust-red raincoat, black trousers and black boots")
            speech = gr.Checkbox(value=False, label="They speak or sing in the clip (lip sync to the source sound)")
        with gr.Group(visible=MODE == "edit"):
            with gr.Row():
                change = gr.Textbox(label="What changes (as it is in the source)", placeholder="the red delivery van parked at the right of the street")
                result = gr.Textbox(label="What it becomes (empty = removed)", placeholder="a blooming cherry tree")
            setting = gr.Textbox(label="The rest of the scene", placeholder="a quiet tree-lined street on a bright morning, with cyclists passing")
        build_btn = gr.Button("✍️ Build the prompt")

    with gr.Accordion("🎭 Optional mask video (Use Source Video with a Mask)", open=False):
        gr.Markdown("White = what may change, black = kept exactly. Source pixels are restored outside the (slightly grown) white area "
                    "after decoding: useful when only one person or object among many changes. Same length and framing as the source.")
        mask = gr.Video(label="Mask video", sources=["upload"], height=200)

    side_by_side = gr.Checkbox(value=True, label="🆚 Also save a side-by-side before / after video")
    with gr.Row():
        gen_btn = gr.Button("🎬 Remake Video", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    with gr.Row():
        video_out = gr.Video(label="🎥 Remade video (original soundtrack)")
        compare_out = gr.Video(label="🆚 Before / after")
    latest_btn = gr.Button("📂 Load latest video (use if the UI lost its connection)")
    status_out = gr.Textbox(label="ℹ️ Status", lines=7, interactive=False)
    deliver_btn = gr.Button("deliver", elem_id="aiq-deliver")  # hidden; clicked by the page script when a result is ready

    _rows, _labels = official_example_rows()
    if _rows:
        gr.Markdown("### 📚 Examples\nViggle's own inputs (source video with sound, painted frames, pins, prompt). Click one to fill "
                    "everything above, then press **🎬 Remake Video**.")
        gr.Examples(_rows, inputs=[source, start_s, duration, pin_on, imgs[0], imgs[1], pins[0], pins[1], prompt, resolution],
                    example_labels=_labels, label=None)
    gr.HTML(FOOTER_HTML)

    for _comp in (source, start_s, duration, resolution, attention):
        _comp.change(source_info, [source, start_s, duration, resolution, attention], source_note, queue=False)
    grab_btn.click(grab_frame_ui, [source, start_s, grab_no, grab_slot, pin_on], [grabbed, grab_note, *pins], queue=False)
    build_btn.click(build_prompt_ui, [duration, pin_on, *imgs, *pins, source_desc, who, name, look, speech, change, result, setting],
                    prompt, queue=False)
    gen_btn.click(start_generation,
                  inputs=[source, start_s, duration, resolution, pin_on, *imgs, *pins, prompt, mask, seed, side_by_side, attention],
                  outputs=status_out, queue=False)
    deliver_btn.click(deliver_result, outputs=[video_out, compare_out, status_out], queue=False, show_progress="hidden")
    stop_btn.click(stop_generation, outputs=status_out, queue=False)
    latest_btn.click(load_latest_video, outputs=[video_out, compare_out, status_out], queue=False)
    clear_btn.click(lambda: (None, None, "Ready.", "", *([None] * MAX_PAINTED), *([None] * MAX_PAINTED), None, -1),
                    outputs=[video_out, compare_out, status_out, prompt, *imgs, *pins, mask, seed], queue=False)


# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# Started without preexec_fn: that would force a real fork() of this process (Python otherwise uses vfork), and a
# fork has to copy every pinned page at once. Step 2 stops leftover tunnels before each start. The link works once
# cloudflared has registered a connection with Cloudflare's edge and the random *.trycloudflare.com name is in DNS;
# --protocol http2 because Kaggle often blocks QUIC.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if found.get("late"):  # the ready block was already printed without it
                    print(f"🌐 Cloudflare: {found['url']}")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found


cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, max_threads=8, allowed_paths=[OUTPUT_DIR, EXAMPLES_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _video_route():
    """The newest video as a plain file (plays in the browser, with a download option): no Gradio involved."""
    videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
    if not videos:
        return JSONResponse({"error": "no video yet"}, status_code=404)
    name = os.path.basename(videos[-1])
    return FileResponse(videos[-1], media_type="video/mp4", headers={"Content-Disposition": f'inline; filename="{name}"',
                                                                      "Cache-Control": "no-store"})


for _path_name, _route in (("/aiq_progress", _progress_route), ("/aiq_video", _video_route)):
    demo.app.router.routes.insert(0, APIRoute(_path_name, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s to come up before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print(f"✅ Ready in {time.time() - _t_load:.0f}s | GPU free {_gb(_free0)} / {_gb(_free1)} | RAM {_ram_in_use()}")
if cf["ready"]:
    print(f"🌐 Cloudflare: {cf['url']}")
else:
    cf["late"] = True
    if not cf["ready"]:
        print("🌐 Cloudflare: warming up, the link appears here in a minute (Gradio works now)")
    else:
        print(f"🌐 Cloudflare: {cf['url']}")
print(f"🔗 Gradio: {getattr(demo, 'share_url', None) or 'unavailable, use the Cloudflare link'}")
print(f"💾 Videos: {OUTPUT_DIR} (latest also at <link>/aiq_video)")
sys.stdout.flush()
while True:  # keep the server process alive; generation logs stream to the notebook cell
    time.sleep(3600)
'''


with open("run_viggle.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)

import json
if os.path.exists("viggle_settings.json"):  # Step 1's choices, for a Step 2 re-run after a kernel restart
    with open("viggle_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Minimal logs: only the engine's own status lines (they start with these markers) and its live progress line reach
# this cell. Library chatter is dropped; Python tracebacks and crash reports are always shown in full.
MARKERS = ("✅", "❌", "⚠️", "🎬", "🌐", "🔗", "💾", "⏹️", "🛑", "🧹", "⚙️", "⏳", "⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏")
CRASH_START = ("Traceback (most recent call last)", "Fatal Python error", "Thread 0x", "Current thread 0x")
_in_crash = [False]


def _keep(part):
    text = part.strip()
    if not text:  # carriage returns / spaces that clear the live line
        return True
    if text.startswith(CRASH_START):
        _in_crash[0] = True
        return True
    if _in_crash[0]:
        if part[:1] in (" ", "\t") or text.startswith(CRASH_START):
            return True
        _in_crash[0] = False  # the exception line closes the traceback
        return True
    return text.startswith(MARKERS)


env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           TF_CPP_MIN_LOG_LEVEL="3", USE_TF="0", USE_FLAX="0", GRADIO_ANALYTICS_ENABLED="False")

print(f"🚀 Starting Viggle Remake ({os.environ.get('VR_MODE', 'swap')} mode)...")
import psutil


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and ~23 GB of RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_viggle.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es): GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass
_kernel_gb = psutil.Process().memory_info().rss / 1024**3
if _kernel_gb > 2.0:
    print(f"⚠️ This notebook kernel holds {_kernel_gb:.1f} GB of RAM. If the engine runs out of memory: Run → Restart, then run Step 2 only.")


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_viggle.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                        preexec_fn=_die_with_kernel)
import threading
import time
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if _keep(part))
        if pending and pending.strip().startswith(MARKERS):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---